# V4 Lab Notebook Parser - Standalone Edition
This notebook contains the entire codebase flattened into executable cells in Google Colab.

## 1. Setup
Upload your image (e.g., `Example_lab_notebook_page.jpg`) to Google Colab. Then, run this cell to install dependencies.

In [1]:
!pip install opencv-python>=4.8 numpy>=1.24 Pillow>=10.0,<12.0 torch torchvision accelerate transformers qwen-vl-utils pix2tex DECIMER>=2.4 chemdataextractor2 rdkit pandas==2.2.2 requests==2.32.4 matplotlib scipy

/bin/bash: line 1: 12.0: No such file or directory


## 2. Codebase
The following cells contain the classes and functions for the parser, loaded in dependency order.

In [2]:
# ==========================================
# Original File: lab_notebook_parser/utils.py
# ==========================================

from pathlib import Path
from typing import Dict, Any, List
import json
import cv2
import matplotlib.pyplot as plt
from PIL import Image


def show_image(img, title="Image", figsize=(10, 8), cmap="gray"):
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def load_image_cv2(image_path: str):
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")
    return img


def save_json(data: Dict[str, Any], path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON to: {path}")


def make_resized_copy_for_vlm(image_path: str, max_width: int = 1600, output_dir: str = "lab_parser_outputs_v3") -> str:
    image_path = Path(image_path)
    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if w <= max_width:
        out_path = output_dir / f"{image_path.stem}_vlm_input.png"
        img.save(out_path)
        return str(out_path)

    scale = max_width / w
    new_size = (max_width, int(h * scale))
    resized = img.resize(new_size)
    out_path = output_dir / f"{image_path.stem}_vlm_input_{max_width}px.png"
    resized.save(out_path)
    return str(out_path)


def crop_image_region(image_path: str, bbox: List[int], output_path: str, padding: int = 8) -> str:
    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(W, x2 + padding)
    y2 = min(H, y2 + padding)

    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"Invalid crop bbox after clipping: {bbox}")

    crop = img.crop((x1, y1, x2, y2))
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    crop.save(output_path)
    return str(output_path)


def extract_json_from_text(text: str) -> Dict[str, Any]:
    import re

    if text is None:
        return {"parse_error": True, "raw_response": None}

    raw = str(text).strip()
    cleaned = raw.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    obj_start = cleaned.find("{")
    obj_end = cleaned.rfind("}")

    if obj_start != -1 and obj_end != -1 and obj_end > obj_start:
        try:
            return json.loads(cleaned[obj_start:obj_end + 1])
        except Exception:
            pass

    return {"parse_error": True, "raw_response": raw}


JSON_REPAIR_PROMPT_TEMPLATE = """
The following output was supposed to be valid JSON, but it may be malformed.

Repair it into valid JSON only. Do not add new information. If an item is incomplete, omit it.

Expected schema:
{schema}

Malformed output:
{raw_output}
"""


def ensure_json_response(model, raw_text: str, schema_hint: str, allow_repair: bool = True) -> Dict[str, Any]:
    parsed = extract_json_from_text(raw_text)

    if not parsed.get("parse_error"):
        return parsed

    if not allow_repair:
        return parsed

    print("JSON parse failed. Asking model to repair output...")

    repair_prompt = JSON_REPAIR_PROMPT_TEMPLATE.format(
        schema=schema_hint,
        raw_output=str(raw_text)[:12000],
    )

    try:
        repaired_raw = model.ask_text(repair_prompt, max_new_tokens=1024)
        repaired = extract_json_from_text(repaired_raw)
        if not repaired.get("parse_error"):
            repaired["_repaired_from_malformed_json"] = True
            return repaired
        return {"parse_error": True, "raw_response": raw_text, "repair_attempt_raw": repaired_raw}
    except Exception as e:
        return {"parse_error": True, "raw_response": raw_text, "repair_error": str(e)}


In [3]:
# ==========================================
# Original File: lab_notebook_parser/prompts.py
# ==========================================

"""
Prompt templates and builders for the V4 lab notebook parser.

Design principles:
  - Every prompt is narrow: one task, one schema
  - Provide explicit symbol guidance (the page's own shorthand)
  - Chain-of-thought experiment reasoning is split into 6 sequential steps
  - Use concrete few-shot examples from the chemistry domain
"""

# ---------------------------------------------------------------------------
# Layout prompt (fallback / hybrid mode)
# ---------------------------------------------------------------------------

LAYOUT_SCHEMA = """\
{
  "regions": [
    {
      "region_id": "r1",
      "type": "heading | text | table | structure_drawing | formula | observation | unknown",
      "bbox": [x1, y1, x2, y2],
      "description": "short description of visible content",
      "confidence": 0.0
    }
  ],
  "notes": []
}"""

LAYOUT_PROMPT = f"""\
You are parsing a scanned handwritten chemistry lab notebook page.

Identify meaningful visual regions on the page.

Return only valid JSON in this schema:

{LAYOUT_SCHEMA}

Rules:
- Use pixel coordinates relative to the provided image.
- Every region_id must be unique: r1, r2, r3, ...
- Prefer larger full-width regions over tiny left-margin strips.
- For each handwritten row, bbox should span the full writing area (not just the margin date).
- Separate molecular drawings, calculations, reaction schemes, and tables when visible.
- Ignore page borders, notebook ruling lines, and dark corner markers.
- Do not invent content. Do not copy schema placeholders.
- Types: "structure_drawing" for ring/bond diagrams; "formula" for equations like Q=I·t
"""

# ---------------------------------------------------------------------------
# Region transcription
# ---------------------------------------------------------------------------

REGION_TRANSCRIPTION_SCHEMA = """\
{
  "lines": [
    {
      "text": "exact handwritten content",
      "normalized_text": "same content with symbols corrected",
      "contains_chemistry": true,
      "uncertain_tokens": ["token1"],
      "confidence": 0.85
    }
  ],
  "notes": []
}"""

REGION_TRANSCRIPTION_PROMPT = f"""\
You are reading one cropped region from a handwritten chemistry lab notebook.

Transcribe ONLY the visible handwritten content. Preserve it exactly.

Return only valid JSON:

{REGION_TRANSCRIPTION_SCHEMA}

Symbol rules — correct these automatically in normalized_text:
  - "22.4C" or "22.4 deg C" → "22.4 °C"
  - "-0.45V" → "-0.45 V"
  - "H2O", "H20" → "H₂O"
  - "Li+" → "Li⁺"
  - "Ag/AgCI" (capital I) → "Ag/AgCl"
  - "mA/cm2", "mA/cm^2" → "mA/cm²"
  - "uL" or "ul" → "μL"
  - "2theta", "2Θ", "2Theta" → "2θ"
  - "->" → "→"
  - "1.5E-4" stays as "1.5E-4" (scientific notation preserved)
  - Preserve: LiTFSI, 12-crown-4, Ag/AgCl, DME, DOL, diglyme, EtOH, TFSI

Critical rules:
  - Do NOT summarise or paraphrase.
  - Do NOT copy schema placeholder text ("transcribed line", "exact line or phrase", etc.).
  - If a token is unclear, add it to uncertain_tokens; still transcribe your best guess.
  - If the crop is blank or unreadable, return: {{"lines": [], "notes": ["unreadable"]}}
  - Do NOT infer formulas from names (e.g. do not write C2F6LiNO4S2 from "LiTFSI").
"""

# ---------------------------------------------------------------------------
# Table transcription
# ---------------------------------------------------------------------------

TABLE_TRANSCRIPTION_SCHEMA = """\
{
  "table": {
    "title": null,
    "columns": ["col1_name", "col2_name"],
    "rows": [
      {"col1_name": "value", "col2_name": "value"}
    ],
    "uncertain_cells": [{"row": 0, "col": "col1_name", "issue": "illegible"}],
    "units": {"col1_name": "°C", "col2_name": "min"},
    "confidence": 0.9
  }
}"""

TABLE_TRANSCRIPTION_PROMPT = f"""\
You are reading one cropped table from a handwritten chemistry lab notebook.

Transcribe the table into structured JSON.

Return only valid JSON:

{TABLE_TRANSCRIPTION_SCHEMA}

Rules:
  - Identify column headers from the first row.
  - Preserve units exactly (°C, min, h, mA, etc.).
  - Put each unit in the "units" dict keyed by column name.
  - Use null for unreadable cells.
  - Do not invent or interpolate missing cells.
  - If uncertain about a cell value, add it to uncertain_cells.
  - Apply the same symbol normalisation as in region transcription (e.g. "30.1C" → "30.1 °C").
"""

# ---------------------------------------------------------------------------
# Chemical structure prompt (for Qwen, used when MolScribe/DECIMER is unavailable)
# ---------------------------------------------------------------------------

STRUCTURE_SCHEMA = """\
{
  "structures": [
    {
      "structure_id": "s1",
      "description": "textual description of the drawing",
      "smiles": null,
      "molecular_formula": null,
      "nearby_labels": ["label text next to or below the structure"],
      "structure_type": "crown_ether | salt | anion | cation | unknown",
      "notes": []
    }
  ]
}"""

STRUCTURE_PROMPT = f"""\
You are analysing a cropped molecular drawing from a handwritten chemistry lab notebook.

Describe what you see and identify the structure if you can.

Return only valid JSON:

{STRUCTURE_SCHEMA}

Rules:
  - Describe the drawing accurately: ring size, bond types, heteroatoms, charge markers.
  - If you can confidently identify a SMILES, provide it — otherwise use null.
  - Do NOT invent SMILES for ambiguous sketches.
  - Capture all nearby text labels (compound names, charge symbols, bracket notations).
  - Example: A ring with 4 oxygens and 4 CH₂ groups = 12-crown-4 = SMILES "C1COCCOCCOCCO1"
  - Example: A Li+ ion inside a crown ether bracket = [Li(12-crown-4)]+
"""

# ---------------------------------------------------------------------------
# Schemas for experiment reasoning steps
# ---------------------------------------------------------------------------

GOAL_SCHEMA = """\
{
  "goal": "exact text of the goal statement",
  "project": "project name or code",
  "date": "date string",
  "evidence_line_ids": ["L1", "L2"]
}"""

MATERIALS_SCHEMA = """\
{
  "materials": [
    {
      "name": "chemical name as written",
      "role": "solute | solvent | electrode | additive | product | reference | other",
      "concentration": "e.g. 1M",
      "volume_or_mass": "e.g. 20 mL",
      "ratio": "e.g. 4:1 v/v",
      "evidence_line_ids": ["L3"]
    }
  ]
}"""

CONDITIONS_SCHEMA = """\
{
  "conditions": [
    {
      "parameter": "e.g. potential",
      "value": "e.g. -0.45 V",
      "reference": "e.g. vs Ag/AgCl",
      "evidence_line_ids": ["L10"]
    }
  ]
}"""

PROCEDURE_SCHEMA = """\
{
  "procedure_steps": [
    {
      "step_number": 1,
      "description": "exact text of step",
      "evidence_line_ids": ["L5", "L6"]
    }
  ]
}"""

RESULTS_SCHEMA = """\
{
  "observations": [
    {
      "text": "exact observed text",
      "measurement_type": "visual | XRD | electrochemical | temperature | other",
      "value": "numeric value if any",
      "unit": "unit if any",
      "evidence_line_ids": ["L20"]
    }
  ],
  "results": [
    {
      "text": "summary of result",
      "evidence_line_ids": ["L21"]
    }
  ]
}"""

CONCLUSION_SCHEMA = """\
{
  "conclusion": {
    "text": null,
    "confidence": 0.0,
    "evidence_line_ids": [],
    "is_inferred": false,
    "removal_reason": null
  },
  "uncertainties": [
    "any ambiguous or unclear items"
  ]
}"""

# ---------------------------------------------------------------------------
# Prompt builders — one per reasoning step
# ---------------------------------------------------------------------------

_SYMBOL_HINT = """\
Symbol key for this notebook page:
  → = reaction arrow
  ⇌ = equilibrium
  °C = degrees Celsius  (may be written as "C" or "deg C")
  μL = microlitre  (may be written as "uL")
  mA/cm² = current density  (may be written as "mA/cm2" or "mA/cm^2")
  2θ = XRD diffraction angle  (may be written as "2theta" or "2Θ")
  Li⁺ = lithium cation
  Ag/AgCl = silver/silver-chloride reference electrode
  RDE = rotating disk electrode
  LiTFSI = lithium bis(trifluoromethanesulfonyl)imide
  12-crown-4 = macrocyclic ether that coordinates Li⁺
  T@ = temperature at a given point (e.g. "→ T@ 22.4°C" means temperature reached 22.4°C)
  ω = rotation speed (e.g. ω = 1600 rpm)
"""


def build_goal_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are extracting the experimental goal from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Return only valid JSON:

{GOAL_SCHEMA}

Critical rules:
  - Quote the goal verbatim from the transcript.
  - Include the project code (e.g. "240604-B") if present.
  - Include the date as written.
  - Cite the exact line IDs as evidence.
  - If no explicit goal is stated, set "goal" to null.

Transcript with line IDs:
{transcript_with_line_ids}
"""


def build_materials_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are extracting the materials list from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Return only valid JSON:

{MATERIALS_SCHEMA}

Critical rules:
  - List every chemical, electrode, and solvent mentioned.
  - Preserve concentrations, volumes, and ratios exactly as written.
  - Assign roles: "solute" for dissolved salts, "solvent" for liquids,
    "electrode" for working/counter/reference electrodes, "additive" for crown ethers etc.
  - Do NOT invent missing quantities.
  - Cite evidence line IDs for each material.

Transcript with line IDs:
{transcript_with_line_ids}
"""


def build_conditions_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are extracting experimental conditions from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Return only valid JSON:

{CONDITIONS_SCHEMA}

Critical rules:
  - Include: applied potential, current density, rotation speed, temperature, time, atmosphere.
  - For potential, always include the reference electrode if stated.
  - Do NOT invent conditions not mentioned in the transcript.
  - Cite evidence line IDs.

Transcript with line IDs:
{transcript_with_line_ids}
"""


def build_procedure_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are reconstructing the experimental procedure from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Return only valid JSON:

{PROCEDURE_SCHEMA}

Critical rules:
  - Order steps chronologically (by line ID order if unclear).
  - Quote each step description verbatim from the transcript.
  - Each step must have at least one evidence_line_id.
  - Separate "prepare electrolyte", "assemble cell", "run deposition", "run characterisation" etc. into distinct steps.

Transcript with line IDs:
{transcript_with_line_ids}
"""


def build_results_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are extracting experimental observations and results from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Return only valid JSON:

{RESULTS_SCHEMA}

Critical rules:
  - Include ALL explicit observations: visual, electrochemical, diffraction, thermal.
  - For XRD: capture peak positions (2θ values) and intensities exactly.
  - For temperature data: extract the full time-vs-temperature table if present.
  - Do NOT infer results not explicitly stated.
  - Cite evidence line IDs for each observation.

Transcript with line IDs:
{transcript_with_line_ids}
"""


def build_conclusion_prompt(transcript_with_line_ids: str,
                             observations_summary: str) -> str:
    return f"""\
You are determining the conclusion of an experiment from a handwritten chemistry lab notebook.

{_SYMBOL_HINT}

Observations already extracted:
{observations_summary}

Return only valid JSON:

{CONCLUSION_SCHEMA}

Critical rules:
  - A conclusion is only valid if explicitly stated in the transcript OR directly supported
    by the extracted observations.
  - If the page records data but no explicit conclusion, set "text" to null.
  - If you infer a conclusion from observations, set "is_inferred" to true and
    "confidence" to at most 0.6.
  - If no supporting evidence_line_ids can be cited, set "text" to null and
    populate "removal_reason".

Transcript with line IDs:
{transcript_with_line_ids}
"""


# ---------------------------------------------------------------------------
# Chemistry extraction from merged transcript (used in all backends)
# ---------------------------------------------------------------------------

CHEMISTRY_SCHEMA = """\
{
  "chemical_entities": [
    {
      "raw_text": "as written in transcript",
      "normalized_name": "canonical chemical name",
      "role": "solute | solvent | electrode | additive | product | other",
      "formula": null,
      "formula_source": null,
      "concentration": null,
      "volume_or_mass": null,
      "evidence": "exact phrase from transcript",
      "notes": []
    }
  ],
  "conditions": [],
  "observations": [],
  "ambiguities": []
}"""


def build_chemistry_from_transcript_prompt(transcript: str) -> str:
    return f"""\
You are extracting chemistry information from a handwritten chemistry lab notebook transcription.

{_SYMBOL_HINT}

Return only valid JSON:

{CHEMISTRY_SCHEMA}

Critical rules:
  - Do NOT invent chemicals not mentioned in the transcript.
  - Do NOT infer molecular formulas from chemical names
    (e.g. do not write "C₂F₆LiNO₄S₂" from "LiTFSI" unless the formula is written).
  - formula_source must be "explicit_page_text" (formula appears verbatim) or null.
  - Use "evidence" to quote the exact phrase containing this chemical.
  - Identify the role of each chemical in the experiment.
  - Capture concentrations, volumes, and ratios from the transcript exactly.

Transcript:
{transcript}
"""

# ---------------------------------------------------------------------------
# Legacy compat: full experiment schema (used in qwen_only backend)
# ---------------------------------------------------------------------------

EXPERIMENT_SCHEMA = """\
{
  "experiment": {
    "project": null,
    "date": null,
    "goal": null,
    "experiment_type": null,
    "materials": [],
    "procedure": [],
    "reaction_or_process": {},
    "observations": [],
    "results": [],
    "conclusion": {
      "text": null,
      "evidence_line_ids": [],
      "confidence": 0.0
    }
  },
  "uncertainties": []
}"""


def build_experiment_from_transcript_prompt(transcript_with_line_ids: str) -> str:
    return f"""\
You are interpreting a handwritten chemistry lab notebook transcription.

{_SYMBOL_HINT}

Return only valid JSON:

{EXPERIMENT_SCHEMA}

Critical rules:
  - Do NOT hallucinate missing product, yield, result, or conclusion.
  - A conclusion is allowed only if explicitly supported by one or more line IDs.
  - If the page records observations but not a final decision, set conclusion.text to null.
  - Every procedure step should include evidence_line_ids.
  - Capture all XRD peak positions, temperatures, and visual observations exactly.

Transcript with line IDs:
{transcript_with_line_ids}
"""


In [4]:
# ==========================================
# Original File: lab_notebook_parser/extraction_rules.py
# ==========================================

"""
Deterministic text-level extraction rules for lab notebook content.

Covers:
  - Scientific text normalisation (units, symbols, Greek letters, sub/superscripts)
  - Quantity extraction (values + units)
  - Chemical formula mention extraction
  - Ratio extraction (v/v, mol%, w/w, etc.)
  - Chemistry NER via ChemDataExtractor (optional)
  - PubChem name resolution
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

# ---------------------------------------------------------------------------
# Data types
# ---------------------------------------------------------------------------


@dataclass
class Quantity:
    raw_text: str
    value: Optional[float]
    unit: str
    quantity_type: str
    source: str
    confidence: float = 0.85
    notes: List[str] = field(default_factory=list)


@dataclass
class FormulaMention:
    raw_text: str
    formula: str
    source: str
    confidence: float = 0.75
    validation: Optional[Dict[str, Any]] = None
    notes: List[str] = field(default_factory=list)


@dataclass
class Ratio:
    raw_text: str
    components: List[str]
    values: List[float]
    ratio_type: str          # "v/v", "w/w", "mol%", "mole_ratio", etc.
    source: str
    confidence: float = 0.8
    notes: List[str] = field(default_factory=list)


@dataclass
class ChemEntity:
    raw_text: str
    normalized_name: Optional[str]
    entity_type: str          # "solute", "solvent", "electrode", "additive", "product", "other"
    formula: Optional[str] = None
    formula_source: Optional[str] = None  # "explicit_text"|"structure_model"|"pubchem_lookup"|"vlm_inferred"
    pubchem_cid: Optional[int] = None
    concentration: Optional[str] = None
    source: str = "ner"
    confidence: float = 0.75
    notes: List[str] = field(default_factory=list)


# ---------------------------------------------------------------------------
# Greek / symbol / sub-superscript normalisation
# ---------------------------------------------------------------------------

# Ordered: longest match first to avoid partial replacements
_GREEK_MAP: List[tuple] = [
    # Theta / XRD angle
    (r"\b2\s*[Tt]heta\b",       "2θ"),
    (r"\b2\s*Θ\b",              "2θ"),
    (r"\btheta\b",              "θ"),
    (r"\bTheta\b",              "θ"),
    # Lambda (wavelength)
    (r"\blambda\b",             "λ"),
    (r"\bLambda\b",             "λ"),
    # Delta
    (r"\bDelta\b",              "Δ"),
    (r"\bdelta\b",              "δ"),
    # Sigma
    (r"\bSigma\b",              "Σ"),
    (r"\bsigma\b",              "σ"),
    # Alpha, beta, gamma
    (r"\balpha\b",              "α"),
    (r"\bbeta\b",               "β"),
    (r"\bgamma\b",              "γ"),
    # Micro prefix
    (r"\bmu\b(?=-?[A-Za-z])",   "μ"),
]

_SYMBOL_MAP: List[tuple] = [
    # Arrows
    (r"->",    " → "),
    (r"=>",    " → "),
    (r"<->",   " ⇌ "),
    (r"<=>",   " ⇌ "),
    # Plus/minus
    (r"\+-",   "±"),
    (r"-\+",   "±"),
    # Degree sign misread as zero or letter O
    (r"(\d)\s*oC\b",            r"\1 °C"),
    (r"(\d)\s*0C\b",            r"\1 °C"),
    (r"(\d)\s*deg\s*[Cc]\b",    r"\1 °C"),
    (r"(\d)\s*°[Cc]\b",         r"\1 °C"),
    # Volume units
    (r"\buL\b",  "μL"),
    (r"\bul\b",  "μL"),
    (r"\bµL\b",  "μL"),
    (r"\bml\b",  "mL"),
    # Electrode shorthand typos
    (r"\bAg/AgCI\b",  "Ag/AgCl"),   # capital I → l
    (r"\bAg/Ag[Cc]l\b",  "Ag/AgCl"),
    # Common misreads
    (r"\bH20\b",  "H₂O"),
    (r"\bH2 0\b", "H₂O"),
    (r"\bH2O\b",  "H₂O"),
    (r"\bLii\+",  "Li⁺"),
    (r"\bLi\+\b", "Li⁺"),
    # Current density
    (r"mA/cm\^?2\b",  "mA/cm²"),
    (r"mA/cm2\b",     "mA/cm²"),
    # Scientific notation: 1.5E-4 → 1.5×10⁻⁴  (preserved as-is, just normalise spacing)
    (r"(\d)\s*[Ee]\s*([+-]?\d+)", r"\1E\2"),
]

# Subscript number map for chemical formulas
_SUB_MAP = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_SUP_MAP = str.maketrans("0123456789+-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻")


def _apply_unicode_subscripts_in_formulas(text: str) -> str:
    """
    Render trailing digits in known chemical formula patterns as Unicode subscripts.
    E.g. LiTFSI: C2F6LiNO4S2 → C₂F₆LiNO₄S₂
    This is applied only to tokens that look like molecular formulas.
    """
    def subscript_formula(m: re.Match) -> str:
        token = m.group(0)
        result = re.sub(r"([A-Z][a-z]?)(\d+)", lambda x: x.group(1) + x.group(2).translate(_SUB_MAP), token)
        # Handle charge markers: Li+, Li2+, etc.
        result = re.sub(r"([A-Z][a-z]?\d*)([+-]\d*)\b", lambda x: x.group(1) + x.group(2).translate(_SUP_MAP), result)
        return result

    # Match likely chemical formulas: start with capital, contain element+digit pattern
    return re.sub(r"\b[A-Z][a-z]?\d*(?:[A-Z][a-z]?\d*)+[+-]?\b", subscript_formula, text)


def normalize_greek_symbols(text: str) -> str:
    for pattern, replacement in _GREEK_MAP:
        text = re.sub(pattern, replacement, text)
    return text


def normalize_symbols(text: str) -> str:
    for pattern, replacement in _SYMBOL_MAP:
        text = re.sub(pattern, replacement, text)
    return text


def normalize_scientific_text(text: str) -> str:
    """
    Full normalisation pipeline:
      1. Whitespace collapse
      2. Greek letter words → Unicode symbols
      3. Arrow / degree / unit symbol fixes
      4. Number+unit spacing
      5. Chemical formula subscript rendering
    """
    if text is None:
        return ""

    s = str(text)
    s = re.sub(r"\s+", " ", s).strip()

    # Greek symbols
    s = normalize_greek_symbols(s)

    # Arrows, degree, electrode names, etc.
    s = normalize_symbols(s)

    # Ensure space between number and unit
    s = re.sub(
        r"(\d)("
        r"mA/cm²|mA/cm2|mA/cm\^2|"
        r"mg|g|kg|mL|μL|uL|L|μM|uM|mM|M|ppm|"
        r"mg/mL|mol|mmol|min|sec|s|hr|h|K|%|"
        r"mA|A|mV|V|C|rpm|cm²|cm2|cm\^2|°C|nm|μm|mm"
        r")\b",
        r"\1 \2",
        s,
    )

    # Chemical subscripts for plain formulas (e.g. H2O → H₂O)
    s = _apply_unicode_subscripts_in_formulas(s)

    return s


# ---------------------------------------------------------------------------
# Quantity extraction
# ---------------------------------------------------------------------------

UNIT_TYPES: Dict[str, str] = {
    "mg": "mass", "g": "mass", "kg": "mass",
    "mL": "volume", "L": "volume", "μL": "volume", "uL": "volume",
    "M": "concentration", "mM": "concentration", "μM": "concentration", "ppm": "concentration",
    "mol": "amount_of_substance", "mmol": "amount_of_substance",
    "°C": "temperature", "K": "temperature",
    "s": "time", "sec": "time", "min": "time", "h": "time", "hr": "time",
    "%": "percentage",
    "mA": "current", "A": "current",
    "mA/cm²": "current_density", "mA/cm2": "current_density", "mA/cm^2": "current_density",
    "V": "potential", "mV": "potential",
    "C": "charge",
    "rpm": "rotation_speed",
    "nm": "length", "μm": "length", "mm": "length", "cm": "length",
    "mol%": "mole_fraction",
    "w/w": "weight_fraction",
    "v/v": "volume_fraction",
}

QUANTITY_PATTERN = re.compile(
    r"""
    (?<![A-Za-z])
    (?P<value>[-+]?(?:\d+\.?\d*|\.\d+)(?:\s*[Ee]\s*[-+]?\d+)?)
    \s*
    (?P<unit>
        mA\s*/\s*cm(?:²|2|\^2)|
        mol%|v/v|w/w|
        °C|μL|uL|mL|L|μM|uM|mM|M|ppm|
        mg/mL|mg|g|kg|mmol|mol|min|sec|s|hr|h|K|%|
        mA|A|mV|V|C|rpm|nm|μm|mm|cm
    )
    (?![A-Za-z])
    """,
    re.VERBOSE,
)


def parse_numeric_value(raw: str) -> Optional[float]:
    if raw is None:
        return None
    s = str(raw).replace("−", "-").replace(" ", "")
    try:
        return float(s)
    except Exception:
        return None


def normalize_unit(unit: str) -> str:
    unit = str(unit).strip().replace(" ", "")
    _MAP = {
        "uL": "μL", "ul": "μL",
        "ml": "mL",
        "mA/cm2": "mA/cm²", "mA/cm^2": "mA/cm²",
    }
    return _MAP.get(unit, unit)


def extract_quantities(text: str, source: str = "merged_transcription") -> List[Quantity]:
    normalized = normalize_scientific_text(text)
    quantities: List[Quantity] = []

    for match in QUANTITY_PATTERN.finditer(normalized):
        left_context = normalized[max(0, match.start() - 30):match.start()].lower()
        # Skip page number references
        if "page" in left_context or "pg" in left_context:
            continue

        raw_value = match.group("value")
        unit = normalize_unit(match.group("unit"))
        raw_text = match.group(0)

        quantities.append(Quantity(
            raw_text=raw_text,
            value=parse_numeric_value(raw_value),
            unit=unit,
            quantity_type=UNIT_TYPES.get(unit, "unknown"),
            source=source,
        ))

    return quantities


# ---------------------------------------------------------------------------
# Ratio extraction  (e.g. "4:1 v/v", "5 mol%")
# ---------------------------------------------------------------------------

RATIO_PATTERN = re.compile(
    r"""
    (?P<a>\d+(?:\.\d+)?)\s*:\s*(?P<b>\d+(?:\.\d+)?)     # a:b ratio
    \s*(?P<type>v/v|w/w|w/v|mol/mol)?                   # optional type label
    |
    (?P<pct>\d+(?:\.\d+)?)\s*(?P<pct_type>mol%|wt%|vol%) # percentage form
    """,
    re.VERBOSE | re.IGNORECASE,
)


def extract_ratios(text: str, source: str = "merged_transcription") -> List[Ratio]:
    ratios: List[Ratio] = []
    normalized = normalize_scientific_text(text)

    for m in RATIO_PATTERN.finditer(normalized):
        if m.group("a") is not None:
            a, b = float(m.group("a")), float(m.group("b"))
            rtype = m.group("type") or "ratio"
            ratios.append(Ratio(
                raw_text=m.group(0).strip(),
                components=[],
                values=[a, b],
                ratio_type=rtype,
                source=source,
            ))
        elif m.group("pct") is not None:
            pct = float(m.group("pct"))
            pct_type = m.group("pct_type")
            ratios.append(Ratio(
                raw_text=m.group(0).strip(),
                components=[],
                values=[pct],
                ratio_type=pct_type,
                source=source,
            ))

    return ratios


# ---------------------------------------------------------------------------
# Chemical formula extraction
# ---------------------------------------------------------------------------

# Full periodic table (118 elements — avoids the old 14-element whitelist bug)
_ELEMENT_SYMBOLS = {
    "H","He","Li","Be","B","C","N","O","F","Ne","Na","Mg","Al","Si","P","S",
    "Cl","Ar","K","Ca","Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Ga",
    "Ge","As","Se","Br","Kr","Rb","Sr","Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd",
    "Ag","Cd","In","Sn","Sb","Te","I","Xe","Cs","Ba","La","Ce","Pr","Nd","Pm",
    "Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu","Hf","Ta","W","Re","Os",
    "Ir","Pt","Au","Hg","Tl","Pb","Bi","Po","At","Rn","Fr","Ra","Ac","Th","Pa",
    "U","Np","Pu","Am","Cm","Bk","Cf","Es","Fm","Md","No","Lr","Rf","Db","Sg",
    "Bh","Hs","Mt","Ds","Rg","Cn","Nh","Fl","Mc","Lv","Ts","Og",
}

FORMULA_PATTERN = re.compile(r"(?<![a-zA-Z])(?:[A-Z][a-z]?\d*){2,}(?:[+-])?\b")


def validate_formula_string(formula: str) -> Dict[str, Any]:
    tokens = re.findall(r"([A-Z][a-z]?)(\d*)", formula)
    if not tokens:
        return {"valid": False, "reason": "no tokens"}
    invalid = [el for el, _ in tokens if el not in _ELEMENT_SYMBOLS]
    if invalid:
        return {"valid": False, "invalid_elements": invalid}
    return {"valid": True, "tokens": tokens}


def extract_explicit_formula_mentions(text: str,
                                      source: str = "merged_transcription") -> List[FormulaMention]:
    normalized = normalize_scientific_text(text)
    mentions: List[FormulaMention] = []
    seen: set = set()

    for match in FORMULA_PATTERN.finditer(normalized):
        formula = match.group(0)
        # Skip very short tokens without digits or charge markers
        if len(formula) <= 2 and not re.search(r"\d|\+|-", formula):
            continue
        if formula in seen:
            continue
        seen.add(formula)

        validation = validate_formula_string(formula)
        if validation.get("valid"):
            mentions.append(FormulaMention(
                raw_text=formula,
                formula=formula,
                source=source,
                confidence=0.80,
                validation=validation,
                notes=["Formula appears explicitly in the transcription."],
            ))

    return mentions


# ---------------------------------------------------------------------------
# Chemistry NER via ChemDataExtractor (optional)
# ---------------------------------------------------------------------------

def run_chemistry_ner(text: str) -> List[ChemEntity]:
    """
    Extract chemical entity mentions using ChemDataExtractor (CDE).
    Falls back gracefully if CDE is not installed.
    """
    try:
        from chemdataextractor import Document
        from chemdataextractor.doc import Paragraph
    except ImportError:
        return []

    try:
        doc = Document(Paragraph(text))
        entities: List[ChemEntity] = []
        seen: set = set()

        for record in doc.records.serialize():
            names = record.get("names") or []
            labels = record.get("labels") or []
            all_names = list(set(names + labels))
            for name in all_names:
                if not name or name in seen:
                    continue
                seen.add(name)
                entities.append(ChemEntity(
                    raw_text=name,
                    normalized_name=name,
                    entity_type="other",
                    source="chemdataextractor",
                ))

        return entities
    except Exception as e:
        return []


# ---------------------------------------------------------------------------
# PubChem name resolution
# ---------------------------------------------------------------------------

def resolve_chemical_with_pubchem(name: str, timeout: int = 8) -> Dict[str, Any]:
    try:
        import requests
        import urllib.parse
    except ImportError:
        return {"resolved": False, "reason": "requests unavailable"}

    if not name or not str(name).strip():
        return {"resolved": False, "reason": "empty name"}

    encoded = urllib.parse.quote(str(name).strip())
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{encoded}"
        f"/property/MolecularFormula,IUPACName,CanonicalSMILES/JSON"
    )

    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code != 200:
            return {"resolved": False, "reason": f"HTTP {resp.status_code}"}

        props = resp.json().get("PropertyTable", {}).get("Properties", [])
        if not props:
            return {"resolved": False, "reason": "no properties"}

        first = props[0]
        return {
            "resolved": True,
            "cid": first.get("CID"),
            "molecular_formula": first.get("MolecularFormula"),
            "iupac_name": first.get("IUPACName"),
            "canonical_smiles": first.get("CanonicalSMILES"),
            "source": "PubChem PUG REST",
        }
    except Exception as e:
        return {"resolved": False, "reason": str(e)}


In [5]:
# ==========================================
# Original File: lab_notebook_parser/qwen_wrapper.py
# ==========================================

from typing import Optional
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info


class QwenVLExtractor:
    def __init__(self, model_name: str = "Qwen/Qwen2.5-VL-7B-Instruct", max_new_tokens: int = 1024):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens

        print(f"Loading model: {model_name}")

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto",
        )

        self.processor = AutoProcessor.from_pretrained(model_name)
        print("Model loaded.")

    def ask_image(self, image_path: str, prompt: str, max_new_tokens: Optional[int] = None) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        return self._generate(messages, max_new_tokens=max_new_tokens)

    def ask_text(self, prompt: str, max_new_tokens: Optional[int] = None) -> str:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        return self._generate(messages, max_new_tokens=max_new_tokens)

    def _generate(self, messages, max_new_tokens: Optional[int] = None) -> str:
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        kwargs = {"text": [text], "padding": True, "return_tensors": "pt"}
        if image_inputs is not None:
            kwargs["images"] = image_inputs
        if video_inputs is not None:
            kwargs["videos"] = video_inputs

        inputs = self.processor(**kwargs).to(self.model.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens or self.max_new_tokens,
            )

        generated_ids_trimmed = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]

        return self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]


/home/rpx2985/labally-assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# ==========================================
# Original File: lab_notebook_parser/trocr_wrapper.py
# ==========================================

"""
TrOCR wrapper for handwriting-specific OCR.
Uses microsoft/trocr-large-handwritten, which is purpose-trained on
handwritten text and significantly outperforms generic VLMs on dense
cursive/printed handwriting.

Designed to share the GPU with Qwen2.5-VL; loads in bfloat16 to minimise
VRAM usage (~4 GB on H100).
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional, List

import torch
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


_DEFAULT_MODEL = "microsoft/trocr-large-handwritten"


class TrOCRExtractor:
    """
    Thin wrapper around TrOCR for handwritten line OCR.

    Usage:
        trocr = TrOCRExtractor()
        text = trocr.transcribe("/path/to/crop.png")
    """

    def __init__(self,
                 model_name: str = _DEFAULT_MODEL,
                 device: Optional[str] = None):
        self.model_name = model_name

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device

        print(f"[TrOCR] Loading {model_name} on {device}...")
        self.processor = TrOCRProcessor.from_pretrained(model_name)
        self.model = VisionEncoderDecoderModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
        ).to(device)
        self.model.eval()
        print("[TrOCR] Model loaded.")

    def transcribe(self, image_path: str, max_new_tokens: int = 256) -> str:
        """
        Transcribe a single crop image (should be a single text line or a
        small block of ~2-3 lines).

        Returns the decoded text string.
        """
        img = Image.open(image_path).convert("RGB")
        return self._transcribe_pil(img, max_new_tokens=max_new_tokens)

    def transcribe_pil(self, img: Image.Image, max_new_tokens: int = 256) -> str:
        return self._transcribe_pil(img, max_new_tokens=max_new_tokens)

    def _transcribe_pil(self, img: Image.Image, max_new_tokens: int = 256) -> str:
        pixel_values = self.processor(
            images=img, return_tensors="pt"
        ).pixel_values.to(self.device, dtype=torch.bfloat16)

        with torch.no_grad():
            generated_ids = self.model.generate(
                pixel_values,
                max_new_tokens=max_new_tokens,
            )

        text = self.processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0]
        return text.strip()

    def transcribe_batch(self, image_paths: List[str],
                         max_new_tokens: int = 256,
                         batch_size: int = 8) -> List[str]:
        """
        Batch-transcribe multiple crops for efficiency.
        """
        results = []
        imgs = [Image.open(p).convert("RGB") for p in image_paths]

        for i in range(0, len(imgs), batch_size):
            batch = imgs[i:i + batch_size]
            pixel_values = self.processor(
                images=batch, return_tensors="pt", padding=True
            ).pixel_values.to(self.device, dtype=torch.bfloat16)

            with torch.no_grad():
                generated_ids = self.model.generate(
                    pixel_values,
                    max_new_tokens=max_new_tokens,
                )

            texts = self.processor.batch_decode(
                generated_ids, skip_special_tokens=True
            )
            results.extend([t.strip() for t in texts])

        return results


In [7]:
# ==========================================
# Original File: lab_notebook_parser/structure_recognizer.py
# ==========================================

"""
Chemical structure recognition for hand-drawn molecular drawings.

Supports two backends:
  - molscribe: transformer model trained specifically for hand-drawn chemical structures
  - decimer:   CNN-based, handles sketch-style drawings well

Validates output SMILES with RDKit and optionally resolves to canonical
name/formula via PubChem.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Dict, Any, List

from PIL import Image


# ---------------------------------------------------------------------------
# Result type
# ---------------------------------------------------------------------------

@dataclass
class StructureResult:
    region_id: str
    backend: str
    smiles: Optional[str] = None
    confidence: float = 0.0
    is_valid_smiles: bool = False
    canonical_smiles: Optional[str] = None
    molecular_formula: Optional[str] = None
    iupac_name: Optional[str] = None
    pubchem_cid: Optional[int] = None
    nearby_labels: List[str] = field(default_factory=list)
    notes: List[str] = field(default_factory=list)
    raw_output: Optional[str] = None


# ---------------------------------------------------------------------------
# RDKit validation
# ---------------------------------------------------------------------------

def validate_and_canonicalize_smiles(smiles: str) -> Dict[str, Any]:
    """
    Validate SMILES using RDKit. Returns:
      {"valid": bool, "canonical_smiles": str, "molecular_formula": str}
    """
    try:
        from rdkit import Chem
        from rdkit.Chem import rdMolDescriptors
    except ImportError:
        return {"valid": False, "reason": "rdkit not installed"}

    if not smiles or not smiles.strip():
        return {"valid": False, "reason": "empty smiles"}

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"valid": False, "reason": "rdkit rejected SMILES"}

    canonical = Chem.MolToSmiles(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    return {
        "valid": True,
        "canonical_smiles": canonical,
        "molecular_formula": formula,
    }


# ---------------------------------------------------------------------------
# PubChem lookup by SMILES
# ---------------------------------------------------------------------------

def pubchem_lookup_by_smiles(smiles: str, timeout: int = 10) -> Dict[str, Any]:
    try:
        import requests, urllib.parse
    except ImportError:
        return {"resolved": False, "reason": "requests unavailable"}

    encoded = urllib.parse.quote(smiles)
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/"
        f"{encoded}/property/MolecularFormula,IUPACName,CanonicalSMILES/JSON"
    )
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code != 200:
            return {"resolved": False, "reason": f"HTTP {resp.status_code}"}
        props = resp.json().get("PropertyTable", {}).get("Properties", [])
        if not props:
            return {"resolved": False, "reason": "no properties"}
        first = props[0]
        return {
            "resolved": True,
            "cid": first.get("CID"),
            "molecular_formula": first.get("MolecularFormula"),
            "iupac_name": first.get("IUPACName"),
            "canonical_smiles": first.get("CanonicalSMILES"),
        }
    except Exception as e:
        return {"resolved": False, "reason": str(e)}


# ---------------------------------------------------------------------------
# MolScribe backend
# ---------------------------------------------------------------------------

class MolScribeBackend:
    """
    Uses MolScribe (https://github.com/thomas0809/MolScribe) to convert
    hand-drawn chemical structure images to SMILES.
    """

    def __init__(self, device: str = "auto"):
        if device == "auto":
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        from molscribe import MolScribe as _MolScribe
        import huggingface_hub
        ckpt_path = huggingface_hub.hf_hub_download(
            "yujieq/MolScribe", "swin_base_char_aux_1m680k.pth"
        )
        self._model = _MolScribe(ckpt_path, device=device)
        print("[MolScribe] Model loaded.")

    def predict(self, image_path: str) -> Dict[str, Any]:
        result = self._model.predict_image_file(image_path, return_atoms_bonds=False)
        return {
            "smiles": result.get("smiles"),
            "confidence": result.get("confidence", 0.0),
        }


# ---------------------------------------------------------------------------
# DECIMER backend
# ---------------------------------------------------------------------------

class DECIMERBackend:
    """
    Uses DECIMER (https://github.com/Kohulan/DECIMER-Image_Transformer) to
    convert hand-drawn chemical structure images to SMILES.
    """

    def __init__(self):
        from DECIMER import predict_SMILES
        self._predict = predict_SMILES
        print("[DECIMER] Ready.")

    def predict(self, image_path: str) -> Dict[str, Any]:
        smiles = self._predict(str(image_path))
        return {"smiles": smiles, "confidence": 0.7}


# ---------------------------------------------------------------------------
# Main recognizer
# ---------------------------------------------------------------------------

class StructureRecognizer:
    """
    High-level chemical structure recognizer.

    Args:
        backend:    "molscribe" | "decimer" | "both"  (both = cascade)
        device:     torch device string ("cuda" or "cpu")
        use_pubchem: whether to do PubChem lookup after recognition
    """

    def __init__(self,
                 backend: str = "molscribe",
                 device: str = "cuda",
                 use_pubchem: bool = True):
        self.backend_name = backend
        self.use_pubchem = use_pubchem
        self._backends: List[Any] = []

        if backend in ("molscribe", "both"):
            try:
                self._backends.append(("molscribe", MolScribeBackend(device=device)))
            except Exception as e:
                print(f"[StructureRecognizer] MolScribe unavailable: {e}")

        if backend in ("decimer", "both"):
            try:
                self._backends.append(("decimer", DECIMERBackend()))
            except Exception as e:
                print(f"[StructureRecognizer] DECIMER unavailable: {e}")

        if not self._backends:
            print("[StructureRecognizer] WARNING: No structure backend loaded. "
                  "Structures will not be recognised.")

    def recognize(self, image_path: str, region_id: str = "unknown",
                  nearby_labels: Optional[List[str]] = None) -> StructureResult:
        """
        Attempt to recognise a chemical structure in the given image crop.
        Tries backends in order; accepts the first valid SMILES.
        Falls back to the second backend if RDKit validation fails.
        """
        nearby_labels = nearby_labels or []
        result = StructureResult(region_id=region_id, backend="none",
                                 nearby_labels=nearby_labels)

        for name, backend in self._backends:
            try:
                raw = backend.predict(image_path)
            except Exception as e:
                result.notes.append(f"{name} prediction error: {e}")
                continue

            smiles = raw.get("smiles")
            confidence = float(raw.get("confidence", 0.0))
            result.raw_output = smiles
            result.backend = name
            result.confidence = confidence

            if not smiles:
                result.notes.append(f"{name}: returned empty SMILES")
                continue

            # Validate with RDKit
            validation = validate_and_canonicalize_smiles(smiles)
            if validation["valid"]:
                result.smiles = smiles
                result.is_valid_smiles = True
                result.canonical_smiles = validation.get("canonical_smiles")
                result.molecular_formula = validation.get("molecular_formula")

                # PubChem lookup
                if self.use_pubchem and result.canonical_smiles:
                    pc = pubchem_lookup_by_smiles(result.canonical_smiles)
                    if pc.get("resolved"):
                        result.pubchem_cid = pc.get("cid")
                        result.molecular_formula = pc.get("molecular_formula") or result.molecular_formula
                        result.iupac_name = pc.get("iupac_name")
                        result.notes.append(f"PubChem resolved: CID={pc.get('cid')}")
                break  # success — stop trying backends

            else:
                result.notes.append(
                    f"{name}: invalid SMILES ({validation.get('reason', 'unknown')})"
                )

        return result

    def recognize_batch(self, image_paths: List[str],
                        region_ids: Optional[List[str]] = None,
                        nearby_labels_list: Optional[List[List[str]]] = None,
                        ) -> List[StructureResult]:
        """Recognise multiple structure crops."""
        region_ids = region_ids or [f"r{i}" for i in range(len(image_paths))]
        nearby_labels_list = nearby_labels_list or [[] for _ in image_paths]
        return [
            self.recognize(p, rid, labels)
            for p, rid, labels in zip(image_paths, region_ids, nearby_labels_list)
        ]


In [8]:
# ==========================================
# Original File: lab_notebook_parser/preprocess.py
# ==========================================

"""
Stage 0: Image Pre-Processing
Deskew, adaptive contrast enhancement, ruled-line detection, and region upscaling.
Applied before any VLM or OCR model call to maximise handwriting readability.
"""

from __future__ import annotations

import math
from pathlib import Path
from typing import List, Tuple, Optional

import cv2
import numpy as np
from PIL import Image


# ---------------------------------------------------------------------------
# Deskew
# ---------------------------------------------------------------------------

def _skew_angle_hough(gray: np.ndarray) -> float:
    """
    Estimate page skew in degrees using probabilistic Hough line transform.
    Returns angle in degrees (positive = clockwise tilt).
    """
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, math.pi / 180, threshold=100,
                             minLineLength=gray.shape[1] // 4,
                             maxLineGap=20)
    if lines is None:
        return 0.0

    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x2 != x1:
            angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
            # Keep only near-horizontal lines (notebook rules)
            if abs(angle) < 10:
                angles.append(angle)

    if not angles:
        return 0.0

    # Robust median estimate
    return float(np.median(angles))


def deskew_image(img: np.ndarray) -> np.ndarray:
    """
    Rotate img to correct skew.  Returns a new ndarray (BGR or gray).
    Only corrects if |skew| > 0.3 degrees to avoid needless resampling.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    angle = _skew_angle_hough(gray)

    if abs(angle) < 0.3:
        return img

    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return rotated


# ---------------------------------------------------------------------------
# Contrast enhancement
# ---------------------------------------------------------------------------

def enhance_contrast(img: np.ndarray,
                     clip_limit: float = 2.5,
                     tile_grid_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
    """
    Apply CLAHE to the L channel of LAB colour space.
    Preserves colour while boosting local contrast for handwriting.
    """
    if img.ndim == 2:
        # Grayscale
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        return clahe.apply(img)

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_ch = clahe.apply(l_ch)
    return cv2.cvtColor(cv2.merge([l_ch, a_ch, b_ch]), cv2.COLOR_LAB2BGR)


def adaptive_binarize(img: np.ndarray,
                      block_size: int = 31,
                      C: int = 10) -> np.ndarray:
    """
    Sauvola-style adaptive thresholding for thin pencil strokes.
    Returns a binary (0/255) uint8 image suitable for TrOCR.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    binary = cv2.adaptiveThreshold(gray, 255,
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY,
                                   block_size, C)
    return binary


# ---------------------------------------------------------------------------
# Ruled-line detection
# ---------------------------------------------------------------------------

def detect_ruled_lines(img: np.ndarray,
                       min_line_fraction: float = 0.5) -> List[int]:
    """
    Detect horizontal notebook ruling lines.

    Returns a sorted list of y-coordinates (in pixels) of each horizontal rule.
    Only lines that span at least `min_line_fraction` of the image width are kept.

    Strategy:
    1. Convert to grayscale and invert so dark rules become white.
    2. Apply a wide horizontal morphological kernel to isolate horizontal lines.
    3. Find connected components and filter by width.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    h, w = gray.shape

    # Invert + threshold
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological horizontal line extraction
    kernel_len = max(30, int(w * 0.4))
    horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horiz_kernel, iterations=2)

    # Find contours of horizontal line segments
    contours, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    y_coords = []
    for cnt in contours:
        x, y, bw, bh = cv2.boundingRect(cnt)
        if bw >= min_line_fraction * w and bh <= max(5, int(h * 0.01)):
            # Mid-point of the line
            y_coords.append(y + bh // 2)

    return sorted(set(y_coords))


# ---------------------------------------------------------------------------
# Region upscaling
# ---------------------------------------------------------------------------

def upscale_region(crop: np.ndarray, scale: float = 2.0) -> np.ndarray:
    """
    Upscale a cropped region using Lanczos resampling.
    Improves fine-detail legibility for structure drawings and small text.
    """
    if scale <= 1.0:
        return crop
    new_w = int(crop.shape[1] * scale)
    new_h = int(crop.shape[0] * scale)
    return cv2.resize(crop, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)


# ---------------------------------------------------------------------------
# Top-level entry point
# ---------------------------------------------------------------------------

def preprocess_for_vlm(image_path: str,
                        output_dir: str = "lab_parser_outputs_v4",
                        max_width: int = 1600,
                        apply_deskew: bool = True,
                        apply_contrast: bool = True) -> dict:
    """
    Full pre-processing pipeline. Returns a dict with:
      - "rgb_path":      path to contrast-enhanced RGB image (for VLM)
      - "binary_path":  path to binarized grayscale image (for TrOCR)
      - "ruled_lines":  list of y-coordinates of horizontal rules
      - "orig_size":    (width, height) of original image
      - "scale_factor": downscale ratio applied (1.0 if no resize)
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    orig_h, orig_w = img_bgr.shape[:2]
    scale_factor = 1.0

    # Optionally resize to cap VLM input size while preserving aspect ratio
    if orig_w > max_width:
        scale_factor = max_width / orig_w
        new_w = max_width
        new_h = int(orig_h * scale_factor)
        img_bgr = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Deskew
    if apply_deskew:
        img_bgr = deskew_image(img_bgr)

    # Detect ruled lines on (possibly deskewed) image
    ruled_lines = detect_ruled_lines(img_bgr)

    # Contrast enhancement
    if apply_contrast:
        img_rgb_enhanced = enhance_contrast(img_bgr)
    else:
        img_rgb_enhanced = img_bgr

    # Binary version for TrOCR
    binary = adaptive_binarize(img_bgr)

    stem = Path(image_path).stem.replace(" ", "_")
    rgb_path = output_dir / f"{stem}_vlm_input.png"
    bin_path = output_dir / f"{stem}_binary.png"

    # Save as PIL (RGB) for VLM compatibility
    Image.fromarray(cv2.cvtColor(img_rgb_enhanced, cv2.COLOR_BGR2RGB)).save(rgb_path)
    Image.fromarray(binary).save(bin_path)

    return {
        "rgb_path": str(rgb_path),
        "binary_path": str(bin_path),
        "ruled_lines": ruled_lines,
        "orig_size": (orig_w, orig_h),
        "scale_factor": scale_factor,
    }


In [9]:
# ==========================================
# Original File: lab_notebook_parser/layout.py
# ==========================================

from typing import Dict, Any, List
from PIL import Image
import cv2



def repair_layout_regions(layout_json: Dict[str, Any], image_path_vlm: str) -> Dict[str, Any]:
    img = Image.open(image_path_vlm).convert("RGB")
    W, H = img.size

    repaired_regions = []
    seen = set()

    for idx, region in enumerate(layout_json.get("regions", []), start=1):
        region = dict(region)
        bbox = region.get("bbox")

        if not bbox or len(bbox) != 4:
            continue

        x1, y1, x2, y2 = [int(v) for v in bbox]
        x1, x2 = sorted([x1, x2])
        y1, y2 = sorted([y1, y2])

        width = x2 - x1
        region["region_id"] = f"r{idx}"

        if width < 0.35 * W:
            x1 = int(0.08 * W)
            x2 = int(0.95 * W)

        pad_y = max(8, int(0.008 * H))
        y1 = max(0, y1 - pad_y)
        y2 = min(H, y2 + pad_y)

        x1 = max(0, x1)
        x2 = min(W, x2)

        if y2 - y1 < 12 or x2 - x1 < 40:
            continue

        repaired_bbox = [x1, y1, x2, y2]
        if tuple(repaired_bbox) in seen:
            continue

        seen.add(tuple(repaired_bbox))
        region["bbox"] = repaired_bbox
        repaired_regions.append(region)

    return {
        "regions": repaired_regions,
        "notes": layout_json.get("notes", []) + [
            "Layout repaired: duplicate IDs fixed; narrow boxes expanded to full writing width."
        ],
        "raw_layout_parse_error": layout_json.get("parse_error", False),
    }


def draw_vlm_regions(image_path: str, layout_regions: List[Dict[str, Any]], figsize=(10, 12), title="VLM Layout Regions"):
    img = load_image_cv2(image_path)
    vis = img.copy()

    for region in layout_regions:
        bbox = region.get("bbox")
        if not bbox or len(bbox) != 4:
            continue

        x1, y1, x2, y2 = [int(v) for v in bbox]
        label = f"{region.get('region_id', '?')}: {region.get('type', '?')}"

        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(vis, label, (x1, max(20, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    show_image(vis, title, figsize=figsize)


In [10]:
# ==========================================
# Original File: lab_notebook_parser/layout_detector.py
# ==========================================

"""
Stage 1: Layout Detection — OpenCV ruled-line segmentation
Replaces the fragile Qwen2.5-VL bbox-prediction layout pass.

Strategy:
  1. Use detected ruled lines to define row bands.
  2. Within each band, use projection profiles to find sub-regions (text vs blank).
  3. Classify each region: text | table | structure_drawing | formula | heading.
  4. Optionally merge with a lightweight VLM region pass for difficult regions.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Optional, Dict, Any, Tuple

import cv2
import numpy as np
from PIL import Image


# ---------------------------------------------------------------------------
# Data types
# ---------------------------------------------------------------------------

REGION_TYPES = {
    "heading", "text", "table", "table_cell",
    "structure_drawing", "reaction_scheme", "molecular_structure",
    "formula", "observation", "unknown"
}


@dataclass
class LayoutRegion:
    region_id: str
    type: str          # one of REGION_TYPES
    bbox: List[int]    # [x1, y1, x2, y2] in scaled image coords
    description: str = ""
    confidence: float = 0.85
    source: str = "opencv"  # "opencv" | "vlm" | "hybrid"
    notes: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _horizontal_projection(gray: np.ndarray) -> np.ndarray:
    """Sum of dark pixel counts per row (after inversion)."""
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary.sum(axis=1).astype(np.float32)


def _vertical_projection(gray: np.ndarray) -> np.ndarray:
    """Sum of dark pixel counts per column."""
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary.sum(axis=0).astype(np.float32)


def _split_by_gaps(projection: np.ndarray,
                   threshold: float,
                   min_run: int = 5) -> List[Tuple[int, int]]:
    """
    Split a projection profile into non-empty segments.
    Returns list of (start, end) index tuples where projection > threshold.
    """
    active = projection > threshold
    segments = []
    in_seg = False
    start = 0
    for i, a in enumerate(active):
        if a and not in_seg:
            start = i
            in_seg = True
        elif not a and in_seg:
            if i - start >= min_run:
                segments.append((start, i))
            in_seg = False
    if in_seg and len(projection) - start >= min_run:
        segments.append((start, len(projection)))
    return segments


def _classify_region(crop_gray: np.ndarray) -> str:
    """
    Heuristic classifier for a cropped region.

    Rules (applied in order):
      - High aspect ratio (wide, short): likely a single text line
      - Has many isolated small connected components arranged in a grid: table
      - Contains large circular/ring structures with few text pixels: structure_drawing
      - Very dense dark pixels with many thin strokes: formula
      - Default: text
    """
    h, w = crop_gray.shape

    # Too small to classify meaningfully
    if h < 8 or w < 20:
        return "unknown"

    _, binary = cv2.threshold(crop_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    density = binary.sum() / (255 * h * w)

    # Structure drawing: moderate density, significant circular features
    circles = cv2.HoughCircles(crop_gray, cv2.HOUGH_GRADIENT, dp=1,
                                minDist=10, param1=50, param2=25,
                                minRadius=8, maxRadius=min(h, w) // 3)
    if circles is not None and len(circles[0]) >= 2 and density < 0.25:
        return "structure_drawing"

    # Table: detect vertical lines (column separators)
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 3)))
    v_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel, iterations=2)
    v_density = v_lines.sum() / (255 * h * w) if h * w > 0 else 0

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 3), 1))
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel, iterations=2)
    h_density = h_lines.sum() / (255 * h * w) if h * w > 0 else 0

    if v_density > 0.01 and h_density > 0.005:
        return "table"

    # Formula: high density with lots of subscript/superscript vertical variation
    vert_proj = _vertical_projection(crop_gray)
    vert_variation = np.std(vert_proj) / (np.mean(vert_proj) + 1e-6)
    if density > 0.08 and vert_variation > 1.5 and h > 40:
        return "formula"

    return "text"


# ---------------------------------------------------------------------------
# Core layout detection
# ---------------------------------------------------------------------------

def detect_layout_from_ruled_lines(img_bgr: np.ndarray,
                                   ruled_lines: List[int],
                                   margin_left_frac: float = 0.08,
                                   margin_right_frac: float = 0.97,
                                   min_row_height: int = 14,
                                   padding: int = 4) -> List[LayoutRegion]:
    """
    Given an image and detected ruled-line y-coordinates, produce layout regions.

    Each pair of consecutive ruled lines defines a "row band".
    Within each band, horizontal projection profiles find active sub-segments.
    """
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Ensure sentinel lines at top and bottom
    ys = sorted(set([0] + ruled_lines + [h]))

    # Content region x-limits
    x1_content = int(margin_left_frac * w)
    x2_content = int(margin_right_frac * w)

    regions: List[LayoutRegion] = []
    rid = 0

    for band_idx in range(len(ys) - 1):
        y_top = ys[band_idx]
        y_bot = ys[band_idx + 1]

        if y_bot - y_top < min_row_height:
            continue

        # Crop the full-width band
        band = gray[y_top:y_bot, x1_content:x2_content]

        # Horizontal projection to find content rows within this band
        horiz_proj = _horizontal_projection(band)
        threshold = max(5.0, float(np.percentile(horiz_proj, 60)))
        row_segs = _split_by_gaps(horiz_proj, threshold, min_run=min_row_height // 2)

        if not row_segs:
            continue

        for row_start, row_end in row_segs:
            abs_y1 = max(0, y_top + row_start - padding)
            abs_y2 = min(h, y_top + row_end + padding)

            if abs_y2 - abs_y1 < min_row_height:
                continue

            # Vertical projection within this row to trim horizontal extent
            row_band = gray[abs_y1:abs_y2, x1_content:x2_content]
            vert_proj = _vertical_projection(row_band)
            v_threshold = max(2.0, float(np.percentile(vert_proj, 50)))
            col_segs = _split_by_gaps(vert_proj, v_threshold, min_run=10)

            if not col_segs:
                col_segs = [(0, x2_content - x1_content)]

            for col_start, col_end in col_segs:
                abs_x1 = max(0, x1_content + col_start - padding)
                abs_x2 = min(w, x1_content + col_end + padding)

                if abs_x2 - abs_x1 < 30:
                    continue

                crop = gray[abs_y1:abs_y2, abs_x1:abs_x2]
                rtype = _classify_region(crop)

                rid += 1
                regions.append(LayoutRegion(
                    region_id=f"r{rid}",
                    type=rtype,
                    bbox=[abs_x1, abs_y1, abs_x2, abs_y2],
                    description="",
                    source="opencv",
                ))

    return regions


def _merge_overlapping_regions(regions: List[LayoutRegion],
                               iou_threshold: float = 0.5) -> List[LayoutRegion]:
    """
    Merge pairs of regions with high IoU (duplicate detection from two sources).
    Prefers VLM-sourced regions over opencv when merging.
    """
    merged: List[LayoutRegion] = []
    used = [False] * len(regions)

    def iou(a: LayoutRegion, b: LayoutRegion) -> float:
        ax1, ay1, ax2, ay2 = a.bbox
        bx1, by1, bx2, by2 = b.bbox
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        area_a = max(1, (ax2 - ax1) * (ay2 - ay1))
        area_b = max(1, (bx2 - bx1) * (by2 - by1))
        return inter / (area_a + area_b - inter)

    for i, r in enumerate(regions):
        if used[i]:
            continue
        group = [r]
        for j in range(i + 1, len(regions)):
            if not used[j] and iou(r, regions[j]) >= iou_threshold:
                group.append(regions[j])
                used[j] = True
        # Pick the best in the group: prefer vlm type annotation
        best = max(group, key=lambda x: (x.source == "vlm", x.confidence))
        merged.append(best)
        used[i] = True

    return merged


def merge_opencv_and_vlm_regions(opencv_regions: List[LayoutRegion],
                                 vlm_regions: List[Dict[str, Any]],
                                 img_size: Tuple[int, int]) -> List[LayoutRegion]:
    """
    Combine OpenCV regions with VLM regions.
    VLM regions are used to:
      - override the region type if confidence is higher
      - add description text
      - add structure_drawing / reaction_scheme regions that OpenCV may miss
    """
    w, h = img_size
    combined: List[LayoutRegion] = list(opencv_regions)

    for idx, vreg in enumerate(vlm_regions):
        bbox = vreg.get("bbox")
        if not bbox or len(bbox) != 4:
            continue

        x1, y1, x2, y2 = [int(v) for v in bbox]
        rtype = vreg.get("type", "unknown")
        if rtype not in REGION_TYPES:
            rtype = "unknown"

        # Only add VLM region if it's for a type that OpenCV typically misses
        if rtype in {"structure_drawing", "reaction_scheme", "molecular_structure", "table"}:
            combined.append(LayoutRegion(
                region_id=f"vlm_{idx+1}",
                type=rtype,
                bbox=[max(0, x1), max(0, y1), min(w, x2), min(h, y2)],
                description=vreg.get("description", ""),
                confidence=float(vreg.get("confidence", 0.7)),
                source="vlm",
            ))

    combined.sort(key=lambda r: r.bbox[1])  # sort top-to-bottom
    merged = _merge_overlapping_regions(combined, iou_threshold=0.4)

    # Re-number
    for i, r in enumerate(merged, start=1):
        r.region_id = f"r{i}"

    return merged


def draw_layout_regions(img_bgr: np.ndarray,
                        regions: List[LayoutRegion],
                        output_path: Optional[str] = None) -> np.ndarray:
    """Draw bounding boxes and labels on a copy of the image."""
    COLOR_MAP = {
        "text": (0, 200, 0),
        "heading": (0, 0, 255),
        "table": (255, 165, 0),
        "table_cell": (200, 200, 0),
        "structure_drawing": (255, 0, 255),
        "reaction_scheme": (200, 0, 200),
        "molecular_structure": (180, 0, 255),
        "formula": (0, 255, 255),
        "observation": (100, 149, 237),
        "unknown": (128, 128, 128),
    }
    vis = img_bgr.copy()
    for r in regions:
        x1, y1, x2, y2 = r.bbox
        color = COLOR_MAP.get(r.type, (128, 128, 128))
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        label = f"{r.region_id}:{r.type[:4]}"
        cv2.putText(vis, label, (x1, max(14, y1 - 2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    if output_path:
        cv2.imwrite(str(output_path), vis)

    return vis


In [11]:
# ==========================================
# Original File: lab_notebook_parser/transcription.py
# ==========================================

"""
Stage 2: Hierarchical transcription with model ensemble.

  2a. Text/special symbols — Qwen2.5-VL + TrOCR ensemble with token voting
  2b. Tables — structured cell-by-cell extraction
  2c. Mathematical formulas — Pix2Tex LaTeX OCR (optional)

The ensemble strategy:
  - Run both Qwen and TrOCR on each text crop
  - If they agree (edit-distance ≤ 2 chars on short tokens), use the agreed text
  - Where they disagree, flag uncertain tokens; prefer Qwen for chemistry abbreviations
    (it understands context), TrOCR for numeric values (it's more precise)
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import cv2
import numpy as np
from PIL import Image



# ---------------------------------------------------------------------------
# Placeholder / quality filters
# ---------------------------------------------------------------------------

BAD_PLACEHOLDER_PHRASES = {
    "transcribed line", "same line with obvious symbols normalized",
    "step description", "exact line or phrase", "as written",
    "short description", "visible content", "text", "normalized_text",
    "exact handwritten content", "same content with symbols corrected",
    "col1_name", "col2_name", "value",
}


def is_bad_placeholder_line(text: Optional[str]) -> bool:
    if text is None:
        return True
    t = str(text).strip().lower()
    if not t:
        return True
    if t in BAD_PLACEHOLDER_PHRASES:
        return True
    if "same line with obvious" in t:
        return True
    if "same content with" in t:
        return True
    return False


def description_looks_like_content(description: Optional[str]) -> bool:
    if description is None:
        return False
    d = str(description).strip()
    if is_bad_placeholder_line(d):
        return False
    useful_patterns = [
        r"\d", r"°C", r"Li", r"Ag/AgCl", r"Goal", r"Electrolyte",
        r"Deposition", r"Apply", r"mA", r"ppm", r"crown", r"glovebox",
        r"temperature", r"XRD", r"Film", r"TFSI", r"glyme", r"EtOH",
        r"diglyme", r"LiTFSI",
    ]
    return any(re.search(p, d, flags=re.IGNORECASE) for p in useful_patterns)


# ---------------------------------------------------------------------------
# Ensemble merge
# ---------------------------------------------------------------------------

def _edit_distance(a: str, b: str) -> int:
    """Simple Levenshtein distance (no weights)."""
    if a == b:
        return 0
    la, lb = len(a), len(b)
    dp = list(range(lb + 1))
    for i in range(1, la + 1):
        new_dp = [i]
        for j in range(1, lb + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            new_dp.append(min(new_dp[j - 1] + 1, dp[j] + 1, dp[j - 1] + cost))
        dp = new_dp
    return dp[lb]


def _merge_two_transcriptions(qwen_text: str,
                               trocr_text: str,
                               prefer_qwen_pattern: str = r"[A-Z][a-z]{1,}[A-Z0-9]") -> Tuple[str, List[str]]:
    """
    Merge Qwen and TrOCR outputs at the token level.

    Rules:
      1. Split both outputs into whitespace tokens.
      2. For each aligned token pair:
         - If identical → accept.
         - If edit distance ≤ 2 and short → accept the Qwen token.
         - If they disagree on a numeric/unit-like token → prefer TrOCR.
         - If they disagree on a chemistry abbreviation → prefer Qwen.
         - Otherwise → flag as uncertain (keep Qwen token).

    Returns (merged_text, uncertain_tokens).
    """
    q_tokens = qwen_text.split()
    t_tokens = trocr_text.split()

    # Align with simple zip (pad shorter)
    max_len = max(len(q_tokens), len(t_tokens))
    q_tokens += [""] * (max_len - len(q_tokens))
    t_tokens += [""] * (max_len - len(t_tokens))

    merged: List[str] = []
    uncertain: List[str] = []

    num_unit_pattern = re.compile(r"^\d+\.?\d*\s*[a-zA-Z°μ%/]+$")
    chem_abbr_pattern = re.compile(prefer_qwen_pattern)

    for qt, tt in zip(q_tokens, t_tokens):
        if not qt and not tt:
            continue
        if qt == tt:
            merged.append(qt)
        elif not tt:
            merged.append(qt)
        elif not qt:
            merged.append(tt)
        else:
            dist = _edit_distance(qt, tt)
            if dist <= 2:
                # Close enough — prefer Qwen (better chemical context)
                merged.append(qt)
            elif num_unit_pattern.match(tt):
                # TrOCR is more reliable for numeric values
                merged.append(tt)
            elif chem_abbr_pattern.search(qt):
                # Qwen better for chemistry abbreviations
                merged.append(qt)
            else:
                # Genuine disagreement
                merged.append(qt)
                uncertain.append(f"{qt}|{tt}")

    return " ".join(merged), uncertain


def transcribe_with_ensemble(qwen,
                              trocr,
                              crop_path: str,
                              qwen_prompt: str,
                              region_type: str = "text",
                              allow_repair: bool = True) -> Dict[str, Any]:
    """
    Transcribe a region crop using Qwen (primary) + TrOCR (secondary).
    Returns a parsed transcription dict with ensemble-merged lines.
    """
    # Qwen pass
    qwen_schema = REGION_TRANSCRIPTION_SCHEMA
    qwen_raw = qwen.ask_image(crop_path, qwen_prompt, max_new_tokens=1024)
    qwen_parsed = ensure_json_response(qwen, qwen_raw, schema_hint=qwen_schema, allow_repair=allow_repair)

    if trocr is None:
        qwen_parsed["_source"] = "qwen_only"
        return qwen_parsed

    # TrOCR pass — use binarized crop for better contrast
    try:
        img_bgr = cv2.imread(crop_path)
        if img_bgr is not None:
            bin_img = adaptive_binarize(img_bgr)
            # Convert binary to 3-channel for TrOCR (expects RGB)
            bin_rgb = cv2.cvtColor(bin_img, cv2.COLOR_GRAY2RGB)
            trocr_text = trocr.transcribe_pil(Image.fromarray(bin_rgb))
        else:
            trocr_text = trocr.transcribe(crop_path)
    except Exception as e:
        trocr_text = ""

    if not trocr_text.strip():
        qwen_parsed["_source"] = "qwen_only_trocr_empty"
        return qwen_parsed

    # Merge each qwen line with the trocr output
    qwen_lines = qwen_parsed.get("lines", [])
    if not qwen_lines:
        # If Qwen gave nothing, use TrOCR as a single line
        return {
            "lines": [{
                "text": trocr_text,
                "normalized_text": normalize_scientific_text(trocr_text),
                "contains_chemistry": bool(re.search(r"Li|Ag|H₂O|mA|M\b|ppm|TFSI|glyme|crown", trocr_text, re.I)),
                "uncertain_tokens": [],
                "confidence": 0.7,
            }],
            "notes": ["qwen_empty_trocr_fallback"],
            "_source": "trocr_fallback",
        }

    merged_lines = []
    for line in qwen_lines:
        qwen_text = line.get("normalized_text") or line.get("text") or ""
        merged_text, uncertain = _merge_two_transcriptions(qwen_text, trocr_text)
        merged_text = normalize_scientific_text(merged_text)
        merged_lines.append({
            "text": line.get("text", ""),
            "normalized_text": merged_text,
            "contains_chemistry": line.get("contains_chemistry"),
            "uncertain_tokens": list(set(line.get("uncertain_tokens", []) + uncertain)),
            "confidence": line.get("confidence", 0.75),
        })

    return {
        "lines": merged_lines,
        "notes": qwen_parsed.get("notes", []) + ["ensemble_qwen_trocr"],
        "_source": "ensemble",
    }


# ---------------------------------------------------------------------------
# Fallback from region description
# ---------------------------------------------------------------------------

def make_fallback_line_from_region_description(region: Dict[str, Any],
                                                crop_path: str) -> Optional[Dict[str, Any]]:
    desc = region.get("description")
    if not description_looks_like_content(desc):
        return None

    norm = normalize_scientific_text(desc)
    return {
        "line_id": f"{region.get('region_id')}_desc",
        "region_id": region.get("region_id"),
        "region_type": region.get("type"),
        "text": desc,
        "normalized_text": norm,
        "contains_chemistry": bool(re.search(
            r"Li|Ag/AgCl|H₂O|mA|M\b|ppm|crown|EtOH|glyme|TFSI|LiTFSI",
            norm, flags=re.IGNORECASE
        )),
        "uncertain_tokens": [],
        "confidence": min(float(region.get("confidence", 0.6) or 0.6), 0.75),
        "crop_path": crop_path,
        "source": "layout_description_fallback",
    }


# ---------------------------------------------------------------------------
# Pix2Tex formula extraction (optional)
# ---------------------------------------------------------------------------

def extract_latex_from_formula_region(crop_path: str) -> Optional[str]:
    """
    Try to extract LaTeX from a formula/equation region using pix2tex.
    Returns LaTeX string or None if unavailable or failed.
    """
    try:
        from pix2tex.cli import LatexOCR
    except ImportError:
        return None

    try:
        model = LatexOCR()
        img = Image.open(crop_path)
        latex = model(img)
        return latex
    except Exception as e:
        return None


# ---------------------------------------------------------------------------
# Main region transcription orchestrator
# ---------------------------------------------------------------------------

def transcribe_regions_with_vlm(qwen,
                                 image_path_vlm: str,
                                 layout_json: Dict[str, Any],
                                 output_dir: str,
                                 allow_repair: bool = True,
                                 trocr=None) -> Dict[str, Any]:
    """
    Transcribe all layout regions.
    Uses ensemble (Qwen + TrOCR) if trocr is not None.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_lines: List[Dict[str, Any]] = []
    tables: List[Dict[str, Any]] = []
    formulas: List[Dict[str, Any]] = []
    raw_region_outputs: List[Dict[str, Any]] = []
    crop_records: List[Dict[str, Any]] = []

    regions = layout_json.get("regions", [])

    for idx, region in enumerate(regions, start=1):
        region_id = region.get("region_id") or f"r{idx}"
        region_type = region.get("type", "unknown")
        bbox = region.get("bbox")

        if not bbox or len(bbox) != 4:
            continue

        safe_rid = str(region_id).replace("/", "_")
        safe_rtype = str(region_type).replace("/", "_")
        crop_path = output_dir / f"{idx:03d}_{safe_rid}_{safe_rtype}.png"
        crop_image_region(image_path_vlm, bbox, crop_path, padding=10)

        crop_records.append({
            "region_id": region_id,
            "region_type": region_type,
            "bbox": bbox,
            "crop_path": str(crop_path),
            "description": region.get("description", ""),
        })

        print(f"  Transcribing {region_id} ({region_type})...")

        # ---- Table regions ----
        if region_type == "table":
            raw = qwen.ask_image(str(crop_path), TABLE_TRANSCRIPTION_PROMPT, max_new_tokens=1024)
            parsed = ensure_json_response(qwen, raw, schema_hint=TABLE_TRANSCRIPTION_SCHEMA, allow_repair=allow_repair)
            raw_region_outputs.append({
                "region_id": region_id, "region_type": region_type,
                "bbox": bbox, "raw_response": raw, "parsed": parsed,
            })
            table = parsed.get("table") if isinstance(parsed, dict) else None
            if table:
                table["region_id"] = region_id
                table["crop_path"] = str(crop_path)
                tables.append(table)
            continue

        # ---- Formula/equation regions ----
        if region_type == "formula":
            latex = extract_latex_from_formula_region(str(crop_path))
            if latex:
                formulas.append({
                    "region_id": region_id,
                    "latex": latex,
                    "crop_path": str(crop_path),
                    "source": "pix2tex",
                })
                # Also add as a text line for the transcript
                all_lines.append({
                    "line_id": f"{region_id}_l1",
                    "region_id": region_id,
                    "region_type": "formula",
                    "text": latex,
                    "normalized_text": normalize_scientific_text(latex),
                    "contains_chemistry": True,
                    "uncertain_tokens": [],
                    "confidence": 0.75,
                    "crop_path": str(crop_path),
                    "source": "pix2tex",
                })
            # Also run Qwen on formula regions as text
            # (fall through to text processing below)

        # ---- Text/heading/observation/formula (Qwen ± TrOCR) ----
        if region_type not in {"structure_drawing", "molecular_structure", "reaction_scheme"}:
            parsed = transcribe_with_ensemble(
                qwen, trocr, str(crop_path),
                qwen_prompt=REGION_TRANSCRIPTION_PROMPT,
                region_type=region_type,
                allow_repair=allow_repair,
            )

            raw_region_outputs.append({
                "region_id": region_id,
                "region_type": region_type,
                "bbox": bbox,
                "description": region.get("description", ""),
                "crop_path": str(crop_path),
                "parsed": parsed,
            })

            good_lines_for_region: List[Dict[str, Any]] = []

            if not parsed.get("parse_error"):
                for i, line in enumerate(parsed.get("lines", [])):
                    text = line.get("text") or ""
                    norm = line.get("normalized_text") or text
                    norm = normalize_scientific_text(norm)

                    if is_bad_placeholder_line(text) or is_bad_placeholder_line(norm):
                        continue

                    good_lines_for_region.append({
                        "line_id": f"{region_id}_l{i+1}",
                        "region_id": region_id,
                        "region_type": region_type,
                        "text": text,
                        "normalized_text": norm,
                        "contains_chemistry": line.get("contains_chemistry"),
                        "uncertain_tokens": line.get("uncertain_tokens", []),
                        "confidence": line.get("confidence"),
                        "crop_path": str(crop_path),
                        "source": parsed.get("_source", "qwen"),
                    })

            if not good_lines_for_region:
                fallback = make_fallback_line_from_region_description(region, str(crop_path))
                if fallback:
                    good_lines_for_region.append(fallback)

            all_lines.extend(good_lines_for_region)

    return {
        "lines": all_lines,
        "tables": tables,
        "formulas": formulas,
        "raw_region_outputs": raw_region_outputs,
        "crop_records": crop_records,
    }


# ---------------------------------------------------------------------------
# Build merged transcript
# ---------------------------------------------------------------------------

def build_transcript_blocks(lines: List[Dict[str, Any]]) -> Tuple[str, str]:
    """
    Build two representations from transcribed lines:
      1. Plain merged text (for NER/chemistry extraction)
      2. Line-ID-annotated text (for evidence-cited experiment reasoning)
    """
    plain_parts: List[str] = []
    line_id_parts: List[str] = []

    for i, line in enumerate(lines, start=1):
        lid = line.get("line_id") or f"line_{i}"
        text = line.get("normalized_text") or line.get("text") or ""
        text = normalize_scientific_text(text)

        if is_bad_placeholder_line(text):
            continue

        plain_parts.append(text)
        line_id_parts.append(f"{lid}: {text}")

    return "\n".join(plain_parts), "\n".join(line_id_parts)


In [12]:
# ==========================================
# Original File: lab_notebook_parser/parser.py
# ==========================================

"""
V4 Lab Notebook Parser — main orchestration.

Three backends selectable via --backend flag:
  qwen_only  : V3-compatible, Qwen only (no TrOCR / MolScribe / DECIMER)
  ensemble   : Qwen + TrOCR text ensemble, still no structure models
  full       : complete V4 pipeline (all models, PubChem, MolScribe/DECIMER)

The parse() method returns a single JSON-serialisable dict with all extracted
information across all four evaluation levels:
  Level 1 — plain text
  Level 2 — special symbols
  Level 3 — chemistry (entities, formulas, structures with SMILES)
  Level 4 — experiment (goal, materials, conditions, procedure, results, conclusion)
"""

from __future__ import annotations

import json
import time
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any, Optional


import cv2


# ---------------------------------------------------------------------------
# Structure interpretation (Qwen fallback, or MolScribe/DECIMER when available)
# ---------------------------------------------------------------------------

def interpret_structures(qwen,
                         image_path_vlm: str,
                         layout_regions,
                         output_dir: Path,
                         allow_repair: bool = True,
                         structure_recognizer=None) -> Dict[str, Any]:
    """
    For each structure_drawing / reaction_scheme / molecular_structure region:
      1. Crop and (optionally) upscale the region
      2. Run MolScribe/DECIMER if available → validated SMILES
      3. Run Qwen STRUCTURE_PROMPT for textual description + nearby labels
      4. Merge results

    Returns {"structures": [...], "raw_structure_outputs": [...]}
    """
    import cv2

    output_dir.mkdir(parents=True, exist_ok=True)
    all_structures = []
    raw_outputs = []

    for idx, region in enumerate(layout_regions, start=1):
        if hasattr(region, 'to_dict'):
            reg_dict = region.to_dict()
        else:
            reg_dict = dict(region)

        rtype = reg_dict.get("type", "unknown")
        if rtype not in {"structure_drawing", "molecular_structure", "reaction_scheme"}:
            continue

        bbox = reg_dict.get("bbox")
        if not bbox or len(bbox) != 4:
            continue

        region_id = reg_dict.get("region_id", f"s{idx}")
        safe_rid = str(region_id).replace("/", "_")
        safe_rtype = str(rtype).replace("/", "_")

        # Crop at 2× resolution for structure models
        crop_path = output_dir / f"{idx:03d}_{safe_rid}_{safe_rtype}.png"
        crop_image_region(image_path_vlm, bbox, crop_path, padding=15)

        # Upscale for clearer recognition
        img_bgr = cv2.imread(str(crop_path))
        if img_bgr is not None:
            upscaled = upscale_region(img_bgr, scale=2.0)
            upscaled_path = output_dir / f"{idx:03d}_{safe_rid}_{safe_rtype}_2x.png"
            cv2.imwrite(str(upscaled_path), upscaled)
        else:
            upscaled_path = crop_path

        print(f"  Interpreting structure in {region_id} ({rtype})...")

        # ---- MolScribe / DECIMER pass ----
        structure_result = None
        if structure_recognizer is not None:
            nearby_labels = [reg_dict.get("description", "")]
            sr = structure_recognizer.recognize(
                str(upscaled_path),
                region_id=region_id,
                nearby_labels=nearby_labels,
            )
            structure_result = {
                "backend": sr.backend,
                "smiles": sr.smiles,
                "is_valid_smiles": sr.is_valid_smiles,
                "canonical_smiles": sr.canonical_smiles,
                "molecular_formula": sr.molecular_formula,
                "iupac_name": sr.iupac_name,
                "pubchem_cid": sr.pubchem_cid,
                "confidence": sr.confidence,
                "notes": sr.notes,
            }

        # ---- Qwen description pass ----
        qwen_raw = qwen.ask_image(str(upscaled_path), STRUCTURE_PROMPT, max_new_tokens=1024)
        qwen_parsed = ensure_json_response(qwen, qwen_raw, schema_hint=STRUCTURE_SCHEMA, allow_repair=allow_repair)

        raw_outputs.append({
            "region_id": region_id,
            "region_type": rtype,
            "crop_path": str(crop_path),
            "upscaled_crop_path": str(upscaled_path),
            "structure_model_result": structure_result,
            "qwen_raw_response": qwen_raw,
            "qwen_parsed": qwen_parsed,
        })

        if qwen_parsed.get("parse_error"):
            continue

        for i, s in enumerate(qwen_parsed.get("structures", [])):
            s = dict(s)
            s["structure_id"] = s.get("structure_id") or f"{region_id}_s{i+1}"
            s["region_id"] = region_id
            s["region_type"] = rtype
            s["crop_path"] = str(crop_path)
            # Merge in structure model result
            if structure_result and structure_result.get("is_valid_smiles"):
                if not s.get("smiles"):
                    s["smiles"] = structure_result["smiles"]
                s["canonical_smiles"] = structure_result.get("canonical_smiles")
                s["molecular_formula"] = structure_result.get("molecular_formula")
                s["iupac_name"] = structure_result.get("iupac_name")
                s["pubchem_cid"] = structure_result.get("pubchem_cid")
                s["structure_model_backend"] = structure_result.get("backend")
                s["structure_model_confidence"] = structure_result.get("confidence")
            all_structures.append(s)

        # If Qwen returned no structures but model did, add a minimal record
        if not qwen_parsed.get("structures") and structure_result and structure_result.get("is_valid_smiles"):
            all_structures.append({
                "structure_id": f"{region_id}_s1",
                "region_id": region_id,
                "region_type": rtype,
                "crop_path": str(crop_path),
                "description": reg_dict.get("description", ""),
                "smiles": structure_result["smiles"],
                "canonical_smiles": structure_result.get("canonical_smiles"),
                "molecular_formula": structure_result.get("molecular_formula"),
                "iupac_name": structure_result.get("iupac_name"),
                "pubchem_cid": structure_result.get("pubchem_cid"),
                "structure_model_backend": structure_result.get("backend"),
                "structure_model_confidence": structure_result.get("confidence"),
            })

    return {"structures": all_structures, "raw_structure_outputs": raw_outputs}


# ---------------------------------------------------------------------------
# Chemistry sanitisation
# ---------------------------------------------------------------------------

def sanitize_chemistry(chemistry_json: Dict[str, Any],
                       explicit_formulas,
                       enable_pubchem: bool) -> Dict[str, Any]:
    chemistry = dict(chemistry_json)
    entities = chemistry.get("chemical_entities", [])
    explicit_formula_texts = {f.formula for f in explicit_formulas}
    cleaned = []

    for ent in entities:
        if isinstance(ent, str):
            ent = {"raw_text": ent, "normalized_name": ent}
        elif isinstance(ent, dict):
            ent = dict(ent)
        else:
            continue

        formula = ent.get("formula")
        formula_source = ent.get("formula_source")

        if formula:
            if formula_source == "explicit_page_text" and formula in explicit_formula_texts:
                pass  # Valid — explicitly written on page
            elif formula_source == "structure_model":
                pass  # Valid — from MolScribe/DECIMER with RDKit validation
            elif formula_source == "pubchem_lookup":
                pass  # Valid — externally verified
            else:
                ent.setdefault("notes", [])
                ent["notes"].append("Formula removed: not explicitly supported by page text or verified model.")
                ent["formula"] = None
                ent["formula_source"] = None

        if enable_pubchem:
            name = ent.get("normalized_name") or ent.get("raw_text")
            resolver = resolve_chemical_with_pubchem(name)
            ent["resolver_result"] = resolver
            if resolver.get("resolved") and not ent.get("formula"):
                ent["formula"] = resolver.get("molecular_formula")
                ent["formula_source"] = "pubchem_lookup"
        else:
            ent["resolver_result"] = None

        cleaned.append(ent)

    chemistry["chemical_entities"] = cleaned
    return chemistry


# ---------------------------------------------------------------------------
# Structured experiment reasoning (V4 chain-of-thought)
# ---------------------------------------------------------------------------

def run_structured_experiment_reasoning(qwen,
                                        transcript_with_line_ids: str,
                                        allow_repair: bool = True) -> Dict[str, Any]:
    """
    Run 6 narrow sequential prompts instead of one big experiment prompt.
    Each step is individually parseable and retryable.
    """

    def _ask(prompt: str, schema: str) -> Dict[str, Any]:
        raw = qwen.ask_text(prompt, max_new_tokens=2048)
        return ensure_json_response(qwen, raw, schema_hint=schema, allow_repair=allow_repair)

    print("    4a: Extracting goal / project / date...")
    goal_json = _ask(build_goal_prompt(transcript_with_line_ids), GOAL_SCHEMA)

    print("    4b: Extracting materials list...")
    materials_json = _ask(build_materials_prompt(transcript_with_line_ids), MATERIALS_SCHEMA)

    print("    4c: Extracting experimental conditions...")
    conditions_json = _ask(build_conditions_prompt(transcript_with_line_ids), CONDITIONS_SCHEMA)

    print("    4d: Reconstructing procedure...")
    procedure_json = _ask(build_procedure_prompt(transcript_with_line_ids), PROCEDURE_SCHEMA)

    print("    4e: Extracting observations and results...")
    results_json = _ask(build_results_prompt(transcript_with_line_ids), RESULTS_SCHEMA)

    # Build a brief observations summary for the conclusion prompt
    obs_list = results_json.get("observations", [])
    obs_summary = "\n".join(
        f"  - {o.get('text', '')} (lines: {o.get('evidence_line_ids', [])})"
        for o in obs_list
    ) if obs_list else "  (no observations extracted yet)"

    print("    4f: Evaluating conclusion...")
    conclusion_json = _ask(
        build_conclusion_prompt(transcript_with_line_ids, obs_summary),
        CONCLUSION_SCHEMA,
    )

    return {
        "goal": goal_json,
        "materials": materials_json,
        "conditions": conditions_json,
        "procedure": procedure_json,
        "results": results_json,
        "conclusion": conclusion_json,
    }


# ---------------------------------------------------------------------------
# Main parser class
# ---------------------------------------------------------------------------

class LabNotebookParserV4:
    """
    V4 pipeline:
      Stage 0: Image pre-processing (deskew, contrast, ruled-line detection)
      Stage 1: Layout detection (OpenCV line segmentation + optional VLM hybrid)
      Stage 2: Hierarchical transcription (Qwen ± TrOCR ensemble + Pix2Tex)
      Stage 3: Chemistry extraction (MolScribe/DECIMER + NER + PubChem)
      Stage 4: Structured experiment reasoning (6 narrow prompts)

    Args:
        qwen:                    QwenVLExtractor instance
        output_dir:              directory for all outputs
        backend:                 "qwen_only" | "ensemble" | "full"
        enable_pubchem:          run PubChem name resolution
        allow_json_repair:       retry malformed JSON with repair prompt
    """

    def __init__(self,
                 qwen,
                 output_dir: str = "lab_parser_outputs_v4",
                 backend: str = "full",
                 enable_pubchem: bool = True,
                 allow_json_repair: bool = True):
        self.qwen = qwen
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.backend = backend
        self.enable_pubchem = enable_pubchem
        self.allow_json_repair = allow_json_repair

        # Lazy-load secondary models based on backend
        self._trocr = None
        self._structure_recognizer = None

        if backend in ("ensemble", "full"):
            self._load_trocr()

        if backend == "full":
            self._load_structure_recognizer()

    def _load_trocr(self):
        try:
            self._trocr = TrOCRExtractor()
        except Exception as e:
            print(f"[Parser] TrOCR not available ({e}). Falling back to Qwen-only transcription.")

    def _load_structure_recognizer(self):
        try:
            self._structure_recognizer = StructureRecognizer(
                backend="both",
                use_pubchem=self.enable_pubchem,
            )
        except Exception as e:
            print(f"[Parser] Structure recognizer not available ({e}). Structures will use Qwen description only.")

    # ------------------------------------------------------------------

    def parse(self,
              image_path: str,
              max_vlm_width: int = 1600,
              save_outputs: bool = True,
              visualize_layout: bool = False) -> Dict[str, Any]:

        start_total = time.time()
        image_path_original = str(image_path)

        print(f"\n{'='*60}")
        print(f"V4 Lab Notebook Parser — backend: {self.backend}")
        print(f"Image: {image_path_original}")
        print(f"{'='*60}\n")

        # ----------------------------------------------------------
        # Stage 0: Pre-processing
        # ----------------------------------------------------------
        print("Stage 0/4: Image pre-processing (deskew + contrast + ruled lines)...")
        preproc = preprocess_for_vlm(
            image_path_original,
            output_dir=str(self.output_dir),
            max_width=max_vlm_width,
            apply_deskew=True,
            apply_contrast=True,
        )
        image_path_vlm = preproc["rgb_path"]
        ruled_lines = preproc["ruled_lines"]
        print(f"  Ruled lines detected: {len(ruled_lines)} at y={ruled_lines}")

        # ----------------------------------------------------------
        # Stage 1: Layout detection
        # ----------------------------------------------------------
        print("\nStage 1/4: Layout detection (OpenCV ruled-line segmentation)...")
        img_bgr = cv2.imread(image_path_vlm)
        opencv_regions = detect_layout_from_ruled_lines(
            img_bgr,
            ruled_lines=ruled_lines,
            min_row_height=14,
            padding=6,
        )
        print(f"  OpenCV regions detected: {len(opencv_regions)}")

        # Optional VLM layout pass for structure drawing regions
        vlm_layout_regions = []
        if self.backend == "full":
            print("  Running VLM layout pass for structure drawing regions...")
            raw_vlm_layout = self.qwen.ask_image(
                image_path_vlm, LAYOUT_PROMPT, max_new_tokens=1024
            )
            vlm_layout_parsed = extract_json_from_text(raw_vlm_layout)
            vlm_layout_regions = vlm_layout_parsed.get("regions", [])
            print(f"  VLM layout regions: {len(vlm_layout_regions)}")

        # Merge OpenCV + VLM regions
        layout_regions = merge_opencv_and_vlm_regions(
            opencv_regions,
            vlm_layout_regions,
            img_size=(img_bgr.shape[1], img_bgr.shape[0]),
        )
        print(f"  Final merged regions: {len(layout_regions)}")

        if visualize_layout:
            vis_path = self.output_dir / "layout_debug.png"
            draw_layout_regions(img_bgr, layout_regions, output_path=str(vis_path))
            print(f"  Layout visualisation saved: {vis_path}")

        # Convert to dict for downstream compatibility
        layout_json = {"regions": [r.to_dict() for r in layout_regions]}

        # ----------------------------------------------------------
        # Stage 2: Transcription
        # ----------------------------------------------------------
        print(f"\nStage 2/4: Transcription ({'ensemble' if self._trocr else 'qwen_only'})...")
        transcription = transcribe_regions_with_vlm(
            self.qwen,
            image_path_vlm,
            layout_json,
            output_dir=str(self.output_dir / "region_crops"),
            allow_repair=self.allow_json_repair,
            trocr=self._trocr,
        )
        print(f"  Lines transcribed: {len(transcription.get('lines', []))}")
        print(f"  Tables found: {len(transcription.get('tables', []))}")
        print(f"  Formulas (LaTeX): {len(transcription.get('formulas', []))}")

        # ----------------------------------------------------------
        # Stage 2 (cont): Build merged transcript
        # ----------------------------------------------------------
        print("\nBuilding merged transcript...")
        merged_text, transcript_with_line_ids = build_transcript_blocks(transcription["lines"])
        merged_text = normalize_scientific_text(merged_text)
        print(f"  Transcript length: {len(merged_text)} chars")
        print("  Preview (first 800 chars):")
        print(merged_text[:800])

        # ----------------------------------------------------------
        # Stage 2 (cont): Deterministic extraction
        # ----------------------------------------------------------
        print("\nDeterministic quantity/formula/ratio extraction...")
        quantities = extract_quantities(merged_text, source="v4_transcription")
        explicit_formulas = extract_explicit_formula_mentions(merged_text, source="v4_transcription")
        ratios = extract_ratios(merged_text, source="v4_transcription")
        print(f"  Quantities: {len(quantities)}")
        print(f"  Formula mentions: {len(explicit_formulas)}")
        print(f"  Ratios: {len(ratios)}")

        # ----------------------------------------------------------
        # Stage 3: Chemistry extraction
        # ----------------------------------------------------------
        print("\nStage 3/4: Chemistry extraction...")

        # 3a: Structure recognition
        print("  3a: Chemical structure recognition...")
        structures = interpret_structures(
            self.qwen,
            image_path_vlm,
            layout_regions,
            output_dir=self.output_dir / "structure_crops",
            allow_repair=self.allow_json_repair,
            structure_recognizer=self._structure_recognizer,
        )
        print(f"  Structures found: {len(structures['structures'])}")

        # 3b: Chemistry NER from transcript
        print("  3b: Chemistry NER...")
        ner_entities = run_chemistry_ner(merged_text)
        print(f"  NER entities: {len(ner_entities)}")

        # 3c: VLM chemistry extraction from transcript
        print("  3c: VLM chemistry extraction from transcript...")
        chem_prompt = build_chemistry_from_transcript_prompt(merged_text)
        chem_raw = self.qwen.ask_text(chem_prompt, max_new_tokens=2048)
        chemistry_json = ensure_json_response(
            self.qwen, chem_raw, schema_hint=CHEMISTRY_SCHEMA, allow_repair=self.allow_json_repair
        )
        chemistry_json["_raw_response"] = chem_raw
        chemistry_json = sanitize_chemistry(chemistry_json, explicit_formulas, self.enable_pubchem)

        # Merge NER entities into chemistry output
        ner_ents_dict = [
            {
                "raw_text": e.raw_text,
                "normalized_name": e.normalized_name,
                "role": e.entity_type,
                "formula": e.formula,
                "formula_source": e.formula_source,
                "source": "chemdataextractor",
            }
            for e in ner_entities
        ]
        chemistry_json["ner_entities"] = ner_ents_dict

        # ----------------------------------------------------------
        # Stage 4: Structured experiment reasoning
        # ----------------------------------------------------------
        print("\nStage 4/4: Structured experiment reasoning (6-step chain)...")

        if self.backend == "full":
            experiment_json = run_structured_experiment_reasoning(
                self.qwen,
                transcript_with_line_ids,
                allow_repair=self.allow_json_repair,
            )
        else:
            # Fallback: single large prompt (V3-compatible)
            exp_prompt = build_experiment_from_transcript_prompt(transcript_with_line_ids)
            exp_raw = self.qwen.ask_text(exp_prompt, max_new_tokens=2048)
            experiment_json = ensure_json_response(
                self.qwen, exp_raw, schema_hint=EXPERIMENT_SCHEMA, allow_repair=self.allow_json_repair
            )

        # ----------------------------------------------------------
        # Assemble result
        # ----------------------------------------------------------
        total_time = round(time.time() - start_total, 2)
        print(f"\nTotal time: {total_time}s")

        result = {
            "document_metadata": {
                "source_original": image_path_original,
                "source_vlm_image": image_path_vlm,
                "strategy": f"v4_{self.backend}",
                "processing_status": "completed",
                "backend": self.backend,
                "enable_pubchem": self.enable_pubchem,
                "total_runtime_seconds": total_time,
            },
            "preprocessing": {
                "ruled_lines_y": ruled_lines,
                "orig_size": preproc["orig_size"],
                "scale_factor": preproc["scale_factor"],
            },
            "layout": layout_json,
            "transcription": {
                "lines": transcription.get("lines", []),
                "tables": transcription.get("tables", []),
                "formulas": transcription.get("formulas", []),
                "crop_records": transcription.get("crop_records", []),
            },
            "merged_transcript": merged_text,
            "merged_transcript_with_line_ids": transcript_with_line_ids,
            "deterministic_quantities": [asdict(q) for q in quantities],
            "explicit_formula_mentions": [asdict(f) for f in explicit_formulas],
            "ratios": [asdict(r) for r in ratios],
            "vlm_chemistry": chemistry_json,
            "structures": structures,
            "experiment_summary": experiment_json,
        }

        if save_outputs:
            base = Path(image_path_original).stem.replace(" ", "_")
            out_json = self.output_dir / f"{base}_v4_{self.backend}.json"
            save_json(result, out_json)

            raw_dir = self.output_dir / f"{base}_raw_debug"
            raw_dir.mkdir(exist_ok=True)

            (raw_dir / "region_raw_outputs.json").write_text(
                json.dumps(transcription.get("raw_region_outputs", []), indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
            (raw_dir / "structure_raw_outputs.json").write_text(
                json.dumps(structures.get("raw_structure_outputs", []), indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
            print(f"\nResults saved: {out_json}")
            print(f"Debug outputs: {raw_dir}")

        return result


# ---------------------------------------------------------------------------
# V3 backward-compat alias
# ---------------------------------------------------------------------------

class VLMFirstLabNotebookParserV3(LabNotebookParserV4):
    """
    Backward-compatible alias.
    Defaults to the 'qwen_only' backend to replicate V3 behaviour.
    """
    def __init__(self, qwen,
                 output_dir: str = "lab_parser_outputs_v3",
                 enable_pubchem_resolution: bool = False,
                 allow_json_repair: bool = True):
        super().__init__(
            qwen=qwen,
            output_dir=output_dir,
            backend="qwen_only",
            enable_pubchem=enable_pubchem_resolution,
            allow_json_repair=allow_json_repair,
        )


## 3. Run Pipeline
Configure your parameters and run the extraction.

In [13]:
import json

# Parameters
IMAGE_PATH = 'Example_lab_notebook_page.jpg'
MODEL_NAME = 'Qwen/Qwen2.5-VL-3B-Instruct' # Note: 2B does not exist! Use 3B or 7B.
OUTPUT_DIR = 'lab_parser_outputs_v4'
MAX_WIDTH = 1600
BACKEND = 'full' # Options: 'qwen_only', 'ensemble', 'full'
ENABLE_PUBCHEM = True
VISUALIZE = True
ALLOW_JSON_REPAIR = True

# Initialize extractor
qwen = QwenVLExtractor(model_name=MODEL_NAME, max_new_tokens=1024)

# Initialize parser
lab_parser = LabNotebookParserV4(
    qwen=qwen,
    output_dir=OUTPUT_DIR,
    backend=BACKEND,
    enable_pubchem=ENABLE_PUBCHEM,
    allow_json_repair=ALLOW_JSON_REPAIR,
)

# Run parsing
result = lab_parser.parse(
    IMAGE_PATH,
    max_vlm_width=MAX_WIDTH,
    save_outputs=True,
    visualize_layout=VISUALIZE,
)

# Display summary
summary = {
    'document_metadata': result.get('document_metadata'),
    'merged_transcript': result.get('merged_transcript'),
    'deterministic_quantities': result.get('deterministic_quantities'),
    'explicit_formula_mentions': result.get('explicit_formula_mentions'),
    'ratios': result.get('ratios'),
    'structures_found': len(result.get('structures', {}).get('structures', [])),
    'experiment_summary': result.get('experiment_summary'),
    'vlm_chemistry': result.get('vlm_chemistry'),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


Loading model: Qwen/Qwen2.5-VL-3B-Instruct


Loading weights: 100%|██████████| 824/824 [00:01<00:00, 640.07it/s]


Model loaded.
[TrOCR] Loading microsoft/trocr-large-handwritten on cuda...


Loading weights: 100%|██████████| 635/635 [00:00<00:00, 9709.43it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[TrOCR] Model loaded.
[StructureRecognizer] MolScribe unavailable: No module named 'molscribe'
[StructureRecognizer] DECIMER unavailable: No module named 'DECIMER'
[StructureRecognizer] WARNING: No structure backend loaded. Structures will not be recognised.

V4 Lab Notebook Parser — backend: full
Image: Example_lab_notebook_page.jpg

Stage 0/4: Image pre-processing (deskew + contrast + ruled lines)...
  Ruled lines detected: 3 at y=[2, 13, 1096]

Stage 1/4: Layout detection (OpenCV ruled-line segmentation)...
  OpenCV regions detected: 233
  Running VLM layout pass for structure drawing regions...


KeyboardInterrupt: 